# Bacteria chemotaxis, live -- people in coloured clothing steer the colony

A bird's-eye camera (Sony a6000 -> OBS -> **OBS Virtual Camera**) watches people walking
over a floor. Each clothing colour is an interference in the bacteria's world:

| Colour | Meaning | Mechanism |
|---|---|---|
| RED | food target | attractant source (1.0); colony cells on food bloom |
| ORANGE | decoy | attractant source (0.5); pulls, but is not food |
| BLACK | toxin | repellent source; colony cells under black die |
| BLUE | wall | hard physics: cells inside are exactly 0 every step |
| GREEN | booster | local time runs 2x: two updates per step, the teacher's cells mature and age twice as fast |

**No path planning.** The attractant and repellent *diffuse* from their sources (walls
only slow diffusion) and the colony climbs the local gradient **greedily**. Like real
chemotaxis it can be caught at a local maximum -- a U-shaped wall with food behind it
traps it until somebody moves. The global nutrient `n` sets the colony's **biomass**:
the manifold maps `n` to the update rule's weights, and `n` decides how long a cell
lives, so a small `n` is a blob and a large `n` a crawling plume.

Supervision comes from a hand-written, deterministic **teacher** CA (every cell greedy,
aging body) run on the same moving worlds; the NCA learns to imitate it from its own
drifted states, so it stays stable over an unbounded live run.

## Setup

In [3]:
import copy
import json
import math
import os
import time
from dataclasses import asdict, dataclass, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
COLOURS = ("red", "orange", "black", "blue", "green")    ## mask channel order
RED, ORANGE, BLACK, BLUE, GREEN = range(5)
print(f"torch {torch.__version__} | device {DEVICE}")

torch 2.9.1+cpu | device cpu


## Configuration

In [9]:
@dataclass
class Config:
    ## cell state
    channels: int = 16
    grid: int = 48
    alive_threshold: float = 0.1
    fire_rate: float = 1.0      ## deterministic updates: the teacher is deterministic, so a
                                ## stochastic rule can only blur toward it (it did: a grey smear)

    ## update rule / manifold
    hidden: int = 128
    leak_init: float = 0.1
    latent_dim: int = 8
    dna_hidden: int = 64
    env_hidden: int = 64

    ## fields: explicit diffusion c <- c + alpha*lap/8 - k*c, k from the length scale
    diff_alpha: float = 0.9
    ell_attr: float = 10.0      ## attractant length scale (cells)
    ell_rep: float = 3.0        ## repellent length scale -- a local push, not a wall
    decoy_level: float = 0.5    ## orange pins the attractant at this level (red at 1)
    attr_floor: float = 1e-4    ## log-sensing floor: la = 1 + log(A + floor) / log(1/floor)
    wall_perm: float = 0.25     ## walls stop cells, but the chemical seeps through them.
                                ## (fully no-flux walls leave NO local maxima off the
                                ## sources -- greedy ascent would never get trapped)
    field_iters: int = 8        ## relaxation iterations per step (the field LAGS motion)
    init_iters: int = 1500      ## relaxation at episode start

    ## greedy teacher
    rep_weight: float = 0.4     ## utility U = log_attr - rep_weight * R: the colony keeps
                                ## ~6 cells from a toxin and still reaches food behind it
    life_max: int = 48          ## cell lifespan at n = 1 (steps); life(n) = n * life_max
    body_r: int = 2             ## radius of the seed colony
    gain_eps: float = 1e-4      ## a cell divides toward a neighbour only for a gain above this

    ## worlds
    max_people: int = 6
    person_r: tuple = (2, 4)
    person_speed: float = 0.3
    toggle_prob: float = 0.004  ## per person per step: appear / vanish
    colour_probs: tuple = (0.30, 0.20, 0.20, 0.15, 0.15)
    margin: int = 3

    ## training
    epochs: int = 4000
    batch: int = 16         ## launch-bound: 16 costs the same wall time as 8
    lr: float = 2e-3
    scalar_lr: float = 1e-4
    n_lo: float = 0.1
    dagger_prob: float = 0.75   ## per slot per epoch: re-anchor the teacher on the NCA colony
    window: tuple = (16, 32)    ## gradient window length
    loss_every: int = 8
    seed: int = 0
    log_every: int = 100
    eval_every: int = 50

    ## io
    out_dir: str = "outputs"
    ckpt_name: str = "bacteria_live.pth"

    @property
    def perception(self) -> int:
        return 3 * (self.channels + 3)      ## +3 sensed channels: attractant, repellent, food

    @property
    def k_attr(self) -> float:
        return (3 * self.diff_alpha / 8) / self.ell_attr ** 2

    @property
    def k_rep(self) -> float:
        return (3 * self.diff_alpha / 8) / self.ell_rep ** 2


CFG = Config()
os.makedirs(CFG.out_dir, exist_ok=True)

## Fields -- diffusion instead of BFS

Two concentrations, attractant `A` (red pinned at 1, orange at 0.5) and repellent `R`
(black pinned at 1), relaxed by an explicit diffusion-with-decay scheme. Walls stop
the *bacteria* but only slow the *chemical*: it seeps through them at `wall_perm` of the
free rate, like a chemical diffusing through the agar under a ridge. That matters --
with fully no-flux walls the field has no local maxima away from its sources (maximum
principle), so greedy ascent would always find the food and could never be trapped;
with seepage, the far side of a U-wall facing the food becomes a real trap. The scheme is
warm-started every step with only a few iterations, so the chemical **lags** behind
moving people -- the far field of somebody who just walked in is still building up.
Steady-state length scale `ell = sqrt(D/k)` with `D = 3*alpha/8` for the 8-neighbour
Laplacian.

In [10]:
_NBR8 = torch.ones(1, 1, 3, 3)
_NBR8[0, 0, 1, 1] = 0.0


class Fields:
    def __init__(self, cfg, B, H, W, device=DEVICE):
        self.cfg = cfg
        self.c = torch.zeros(B, 2, H, W, device=device)
        self.k = torch.tensor([cfg.k_attr, cfg.k_rep], device=device).view(1, 2, 1, 1)
        self.nbr = _NBR8.to(device).repeat(2, 1, 1, 1)

    def _pins(self, masks):
        """(value, where) for the Dirichlet sources of both channels."""
        red, orange, black = masks[:, RED:RED + 1], masks[:, ORANGE:ORANGE + 1], masks[:, BLACK:BLACK + 1]
        val_a = torch.maximum(red, orange * self.cfg.decoy_level)
        val = torch.cat([val_a, black], 1)
        where = torch.cat([torch.maximum(red, orange), black], 1) > 0
        return val, where

    @torch.no_grad()
    def relax(self, masks, iters, sel=None):
        """`iters` relaxation steps; with `sel` (B,1,1,1) bool only those slots commit."""
        ## Conductance between neighbours i,j: 1 if both free, p = wall_perm if either is a
        ## wall. Then  lap_i = p*conv8(c) + (1-p)*f_i*conv8(f*c) - c_i*G_i  with the total
        ## conductance G_i precomputed. Decay and Dirichlet pins fold into coefficients.
        B, _, H, W = self.c.shape
        p, a = self.cfg.wall_perm, self.cfg.diff_alpha
        f = (1.0 - masks[:, BLUE:BLUE + 1]).expand(-1, 2, -1, -1)
        ones = torch.ones_like(f)
        G = p * F.conv2d(ones, self.nbr, padding=1, groups=2) \
            + (1 - p) * f * F.conv2d(f, self.nbr, padding=1, groups=2)
        val, where = self._pins(masks)
        where &= f > 0                         ## a source standing in a wall is walled in
        keep = (~where).float()
        coef = (1.0 - a * G / 8 - self.k) * keep
        pin = val * where.float()
        g_all, g_free = (a / 8) * p * keep, (a / 8) * (1 - p) * keep * f
        nbr4 = self.nbr.repeat(2, 1, 1, 1)
        c = torch.addcmul(pin, self.c, keep)
        for _ in range(iters):
            s = F.conv2d(torch.cat([c, c * f], 1), nbr4, padding=1, groups=4)
            c = torch.addcmul(torch.addcmul(torch.addcmul(pin, coef, c), g_all, s[:, :2]),
                              g_free, s[:, 2:])
        self.c.copy_(c if sel is None else torch.where(sel, c, self.c))   ## in place: CUDA graphs
        return self.c

    def log_attr(self):
        """(B,1,H,W) attractant as bacteria sense it: LOG concentration (Weber's law --
        E. coli responds to relative change), normalised to [0,1]; 0 below the floor."""
        e = self.cfg.attr_floor
        return (1.0 + torch.log(self.c[:, 0:1] + e) / math.log(1.0 / e)).clamp(0.0, 1.0)

    def sense(self, masks):
        """(B,3,H,W): log-attractant, repellent, raw food mask -- what a cell perceives."""
        return torch.cat([self.log_attr(), self.c[:, 1:2], masks[:, RED:RED + 1]], 1)

## Worlds -- walls and walking people

Training worlds are synthetic: 0-3 static wall shapes (blue) and up to six "people",
discs of radius 2-4 cells with a clothing colour, walking with a slowly turning
heading, bouncing off the borders, and occasionally stepping in or out of view. The
camera produces exactly the same `(B,5,H,W)` masks, so the model cannot tell them apart.

In [11]:
def sample_walls(g, rng):
    """0-3 shapes from {wall with a gap, blob, bar}; the outer 2-cell ring stays free."""
    obs = np.zeros((g, g), bool)
    n_shapes = 0 if rng.random() < 0.25 else int(rng.integers(1, 4))
    for _ in range(n_shapes):
        kind = int(rng.integers(0, 3))
        if kind == 0:                                        ## wall with a gap
            pos, th = int(rng.integers(12, g - 12)), int(rng.integers(2, 4))
            gc, gw = int(rng.integers(8, g - 8)), int(rng.integers(3, 6))
            if rng.random() < 0.5:
                obs[pos:pos + th, :] = True
                obs[pos:pos + th, max(0, gc - gw):gc + gw] = False
            else:
                obs[:, pos:pos + th] = True
                obs[max(0, gc - gw):gc + gw, pos:pos + th] = False
        elif kind == 1:                                      ## blob
            cy, cx = rng.integers(8, g - 8, size=2)
            r = int(rng.integers(3, 7))
            yy, xx = np.mgrid[0:g, 0:g]
            obs |= (yy - cy) ** 2 + (xx - cx) ** 2 <= r * r
        else:                                                ## bar
            cy, cx = rng.integers(8, g - 8, size=2)
            L, th = int(rng.integers(8, 16)), int(rng.integers(2, 4))
            if rng.random() < 0.5:
                obs[max(0, cy - L // 2):cy + L // 2, max(0, cx - th // 2):cx + th // 2 + 1] = True
            else:
                obs[max(0, cy - th // 2):cy + th // 2 + 1, max(0, cx - L // 2):cx + L // 2] = True
    obs[:2, :] = False; obs[-2:, :] = False; obs[:, :2] = False; obs[:, -2:] = False
    return obs


class World:
    """Batched walls + people. `pos`, `vel` (B,P,2) float; `r` (B,P); `col` (B,P) colour
    index; `active` (B,P) bool; `walls` (B,H,W) bool; `seed` (B,2) long."""

    def __init__(self, cfg, walls, pos, vel, r, col, active, seed, device,
                 wander=True, gen_seed=0):
        self.cfg, self.device, self.wander = cfg, device, wander
        self.walls = torch.as_tensor(walls, device=device).bool()
        self.pos = torch.as_tensor(pos, dtype=torch.float32, device=device).clone()
        self.vel = torch.as_tensor(vel, dtype=torch.float32, device=device).clone()
        self.r = torch.as_tensor(r, dtype=torch.float32, device=device)
        self.col = torch.as_tensor(col, dtype=torch.long, device=device)
        self.active = torch.as_tensor(active, device=device).bool().clone()
        self.seed = torch.as_tensor(seed, dtype=torch.long, device=device)
        self.gen = torch.Generator(device=device)
        self.gen.manual_seed(int(gen_seed))
        H, W = self.walls.shape[-2:]
        yy, xx = torch.meshgrid(torch.arange(H, device=device), torch.arange(W, device=device),
                                indexing="ij")
        self._yy, self._xx = yy.float(), xx.float()

    @property
    def B(self):
        return self.walls.shape[0]

    @classmethod
    def random(cls, cfg, B, rng, device=DEVICE):
        g, P = cfg.grid, cfg.max_people
        walls = np.stack([sample_walls(g, rng) for _ in range(B)])
        r = rng.integers(cfg.person_r[0], cfg.person_r[1] + 1, size=(B, P)).astype(np.float32)
        col = rng.choice(5, size=(B, P), p=np.array(cfg.colour_probs))
        col[:, 0] = np.where(rng.random(B) < 0.8, RED, col[:, 0])     ## food is usually there
        pos = rng.uniform(r[..., None], g - 1 - r[..., None], size=(B, P, 2)).astype(np.float32)
        speed = rng.uniform(0, cfg.person_speed, size=(B, P))
        head = rng.uniform(0, 2 * np.pi, size=(B, P))
        vel = np.stack([speed * np.sin(head), speed * np.cos(head)], -1).astype(np.float32)
        n_on = rng.integers(1, P + 1, size=B)
        active = np.arange(P)[None, :] < n_on[:, None]
        w = cls(cfg, walls, pos, vel, r, col, active, np.zeros((B, 2)), device,
                wander=True, gen_seed=int(rng.integers(1 << 31)))
        w.seed = w._pick_seeds(rng)
        return w

    @classmethod
    def scripted(cls, cfg, people, seeds, walls=None, device=DEVICE):
        """people: per batch element, a list of dicts {pos, vel, r, colour}. Straight-line
        motion (bouncing at the borders), nobody appears or vanishes."""
        g, B = cfg.grid, len(people)
        P = max(1, max(len(p) for p in people))
        pos, vel = np.zeros((B, P, 2), np.float32), np.zeros((B, P, 2), np.float32)
        r, col, active = np.full((B, P), 2.0, np.float32), np.zeros((B, P), int), np.zeros((B, P), bool)
        for b, plist in enumerate(people):
            for i, p in enumerate(plist):
                pos[b, i], vel[b, i], r[b, i] = p["pos"], p.get("vel", (0, 0)), p["r"]
                col[b, i], active[b, i] = COLOURS.index(p["colour"]), True
        if walls is None:
            walls = np.zeros((B, g, g), bool)
        return cls(cfg, walls, pos, vel, r, col, active, np.array(seeds), device, wander=False)

    def _pick_seeds(self, rng):
        """A free cell per world: not in a wall, clear of every person by 2 cells."""
        g, m = self.cfg.grid, self.cfg.margin
        d2 = (self._yy - self.pos[..., 0, None, None]) ** 2 + (self._xx - self.pos[..., 1, None, None]) ** 2
        near = ((d2 <= (self.r[..., None, None] + 2) ** 2) & self.active[..., None, None]).any(1)
        bad = (near | self.walls).cpu().numpy()
        bad[:, :m, :] = True; bad[:, -m:, :] = True; bad[:, :, :m] = True; bad[:, :, -m:] = True
        seeds = []
        for b in range(self.B):
            cells = np.argwhere(~bad[b])
            if len(cells) == 0:                     ## crowded: fall back to any free cell
                cells = np.argwhere(~self.walls[b].cpu().numpy())
            seeds.append(cells[rng.integers(len(cells))])
        return torch.as_tensor(np.array(seeds), dtype=torch.long, device=self.device)

    def splice(self, idx, other):
        """Replace slots `idx` with the (len(idx)-slot) world `other`."""
        for k in ("walls", "pos", "vel", "r", "col", "active", "seed"):
            getattr(self, k)[idx] = getattr(other, k)

    def masks(self):
        d2 = (self._yy - self.pos[..., 0, None, None]) ** 2 + (self._xx - self.pos[..., 1, None, None]) ** 2
        inside = ((d2 <= self.r[..., None, None] ** 2) & self.active[..., None, None]).float()
        onehot = (self.col[..., None] == torch.arange(5, device=self.device)).float()
        m = torch.einsum("bphw,bpc->bchw", inside, onehot).clamp_(max=1.0)
        m[:, BLUE] = torch.maximum(m[:, BLUE], self.walls.float())
        return m

    def step(self):
        if self.wander:                             ## slowly turning heading
            dth = torch.randn(self.vel.shape[:2], device=self.device, generator=self.gen) * 0.15
            c, s = torch.cos(dth), torch.sin(dth)
            vy, vx = self.vel[..., 0], self.vel[..., 1]
            self.vel.copy_(torch.stack([c * vy - s * vx, s * vy + c * vx], -1))
            flip = torch.rand(self.active.shape, device=self.device, generator=self.gen) < self.cfg.toggle_prob
            self.active ^= flip
        self.pos += self.vel
        lo = self.r[..., None].expand_as(self.pos)
        hi = self.cfg.grid - 1 - lo
        below, above = self.pos < lo, self.pos > hi
        self.pos.copy_(torch.where(below, 2 * lo - self.pos, torch.where(above, 2 * hi - self.pos, self.pos)))
        self.vel.copy_(torch.where(below | above, -self.vel, self.vel))

## The teacher -- a greedy chemotaxis CA (the supervision)

Every bacterium is greedy, and every rule is local:

- each cell's **greedy choice** is its best neighbour by utility `U = log_attr - rep_weight * R`
  -- but only if it beats staying put by `gain_eps`. Log-sensing (Weber's law) gives a steady
  pull at any distance from the food; the short-range repellent only matters near a toxin.
  Walls are never chosen;
- a **mature** cell (age >= 2) divides into its choice if that cell is free -- so the front
  climbs the gradient at 0.5 cells/step. A cell whose choice is *itself* sits at a local maximum
  of `U`: it is a **tip**, and keeps dividing in place (into its free neighbours). That is how
  greedy ascent gets **trapped**: behind a U-shaped wall the tip just sits in the cup;
- every living cell ages by one per step (two on green -- local time runs twice as fast); a cell
  dies when `age > life(n) = n * life_max`, so `n` is the **biomass**: the length of the trail
  the colony drags behind its front. Dropping `n` kills the tail at once;
- **feeding**: the colony spreads over red food one cell per step and feeding cells don't age,
  so it engulfs the food person and trails after them when they walk away;
- cells under blue (crushed) or black (poisoned) die; if every cell dies the colony is
  **extinct** (the live loop reseeds);
- colour encodes age (bright young front -> dark old tail); cells on red food **bloom**.

An earlier version moved a single "head" cell and stamped a disc around it. Only one cell being
the head is not something a cell can sense locally, so the NCA hedged with a faint fan-shaped
smear; this version has no special cell at all.

In [12]:
CORE = np.array([0.30, 0.42, 0.12])      ## old tail
FRONT = np.array([0.90, 0.95, 0.45])     ## young front
BLOOM = np.array([0.98, 0.45, 0.80])     ## feeding on red food

_OFFS = ((0, 1), (0, -1), (1, 0), (-1, 0), (1, 1), (1, -1), (-1, 1), (-1, -1))   ## tie order: axes first


def life(n, cfg):
    """Cell lifespan in steps at nutrient n (float or tensor)."""
    if torch.is_tensor(n):
        return torch.clamp(torch.round(n * cfg.life_max), min=1)
    return max(1, int(round(float(n) * cfg.life_max)))


def _nbr(x, dy, dx, fill):
    """out[..., y, x] = x[..., y + dy, x + dx], `fill` outside the grid."""
    H, W = x.shape[-2:]
    p = F.pad(x, (1, 1, 1, 1), value=fill)
    return p[..., 1 + dy:1 + dy + H, 1 + dx:1 + dx + W]


class Teacher:
    """The state is just the age grid `age` (B,H,W), inf = dead; `alive` (B,) flags colonies
    that still exist; `t` is a per-slot step counter."""

    def __init__(self, cfg, seed, masks):
        B, _, H, W = masks.shape
        dev = masks.device
        self.cfg = cfg
        self.t = torch.zeros(B, dtype=torch.long, device=dev)
        self.age = torch.full((B, H, W), float("inf"), device=dev)
        self.alive = torch.ones(B, dtype=torch.bool, device=dev)
        yy, xx = torch.meshgrid(torch.arange(H, device=dev), torch.arange(W, device=dev), indexing="ij")
        seed = torch.as_tensor(seed, device=dev).long()
        d2 = (yy - seed[:, 0, None, None]) ** 2 + (xx - seed[:, 1, None, None]) ** 2
        self.age.copy_(torch.where(d2 <= cfg.body_r ** 2, torch.zeros_like(self.age), self.age))
        self._kill_env(masks)

    def _kill_env(self, masks):
        dead = (masks[:, BLUE] > 0) | (masks[:, BLACK] > 0)
        self.age.copy_(torch.where(dead, torch.full_like(self.age, float("inf")), self.age))
        self.alive &= torch.isfinite(self.age).flatten(1).any(1)

    def choice(self, fields, masks):
        """(B,H,W) long: 0 = stay (a local maximum), k = divide toward _OFFS[k-1]."""
        cfg, inf = self.cfg, float("inf")
        wall = masks[:, BLUE] > 0
        U = fields.log_attr()[:, 0] - cfg.rep_weight * fields.c[:, 1]
        U = torch.where(wall, torch.full_like(U, -inf), U)
        best_u, best_k = U + cfg.gain_eps, torch.zeros_like(U, dtype=torch.long)
        for k, (dy, dx) in enumerate(_OFFS, start=1):
            un = _nbr(U, dy, dx, -inf)
            better = un > best_u
            best_u = torch.where(better, un, best_u)
            best_k = torch.where(better, torch.full_like(best_k, k), best_k)
        return best_k

    @torch.no_grad()
    def step(self, fields, masks, n):
        wall, black = masks[:, BLUE] > 0, masks[:, BLACK] > 0
        best_k = self.choice(fields, masks)
        self.age.add_(1.0 + masks[:, GREEN])        ## local time runs twice as fast on green
        alive = torch.isfinite(self.age)
        mature = alive & (self.age >= 2)
        born = torch.zeros_like(alive)
        for k, (dy, dx) in enumerate(_OFFS, start=1):   ## a mature cell divides into its choice
            born |= _nbr((mature & (best_k == k)).float(), -dy, -dx, 0.0) > 0
        tips = alive & (best_k == 0)                    ## local maxima keep dividing in place
        born |= F.max_pool2d(tips[:, None].float(), 3, stride=1, padding=1)[:, 0] > 0
        born &= ~alive & ~wall & ~black
        self.age.copy_(torch.where(born | tips, torch.zeros_like(self.age), self.age))
        ## feeding: the colony spreads over red food one cell per step, and feeding cells
        ## don't age -- it engulfs the food person and trails after them when they walk off
        alive = torch.isfinite(self.age)[:, None].float()
        feed = (F.max_pool2d(alive, 3, stride=1, padding=1)[:, 0] > 0) & (masks[:, RED] > 0)
        self.age.copy_(torch.where(feed, torch.zeros_like(self.age), self.age))
        lf = life(torch.as_tensor(n, dtype=torch.float32, device=self.age.device), self.cfg)
        self.age.copy_(torch.where(self.age > lf.view(-1, 1, 1), torch.full_like(self.age, float("inf")), self.age))
        self._kill_env(masks)
        self.t += 1

    def splice(self, idx, other):
        """Replace slots `idx` with the (len(idx)-slot) teacher `other`."""
        for k in ("t", "age", "alive"):
            getattr(self, k)[idx] = getattr(other, k)

    @torch.no_grad()
    def anchor(self, rgba, masks, sel):
        """DAgger: re-read the slots in `sel` ((B,) bool) from a colony IMAGE -- alive where
        alpha > 0.5 (and not in a wall/toxin), age from the colour along FRONT -> CORE (bloom
        cells count as young). The teacher then continues from the learner's colony."""
        dev = self.age.device
        front = torch.tensor(FRONT, dtype=torch.float32, device=dev).view(1, 3, 1, 1)
        d = torch.tensor(CORE - FRONT, dtype=torch.float32, device=dev).view(1, 3, 1, 1)
        shade = ((rgba[:, :3] - front) * d).sum(1) / (d * d).sum()
        age = torch.round(shade.clamp(0, 1) * self.cfg.life_max)
        age = torch.where(masks[:, RED] > 0, torch.zeros_like(age), age)
        kill = (masks[:, BLUE] > 0) | (masks[:, BLACK] > 0)
        ## the image is drawn THICK (see rgba): erode it back to the thin state, treating
        ## walls/toxins and the border as "inside" so cells beside them are not eroded away
        thick = (rgba[:, 3] > 0.5) | kill
        pad = F.pad(thick[:, None].float(), (1, 1, 1, 1), value=1.0)
        alive = (-F.max_pool2d(-pad, 3, stride=1)[:, 0] > 0) & ~kill & (rgba[:, 3] > 0.5)
        age = torch.where(alive, age, torch.full_like(age, float("inf")))
        ok = sel & alive.flatten(1).any(1)
        self.age.copy_(torch.where(ok[:, None, None], age, self.age))
        self.alive.copy_(torch.where(ok, torch.ones_like(self.alive), self.alive))

    def rgba(self, masks):
        """(B,4,H,W) target image: the colony drawn THICK -- dilated by one cell (a 1-cell
        trail is too thin a target for an NCA to hold), each drawn cell coloured by the
        youngest colony cell next to it, bloom on food, nothing on walls/toxins."""
        big = torch.full_like(self.age, 1e6)
        age = -F.max_pool2d(-torch.where(torch.isfinite(self.age), self.age, big)[:, None], 3,
                            stride=1, padding=1)[:, 0]
        kill = (masks[:, BLUE] > 0) | (masks[:, BLACK] > 0)
        alive = (age < 1e6) & ~kill
        shade = (age / self.cfg.life_max).clamp(0, 1)
        front = torch.tensor(FRONT, dtype=torch.float32, device=self.age.device).view(1, 3, 1, 1)
        core = torch.tensor(CORE, dtype=torch.float32, device=self.age.device).view(1, 3, 1, 1)
        bloom = torch.tensor(BLOOM, dtype=torch.float32, device=self.age.device).view(1, 3, 1, 1)
        rgb = front + (core - front) * shade[:, None]
        rgb = torch.where((masks[:, RED:RED + 1] > 0), bloom, rgb)
        a = alive[:, None].float()
        return torch.cat([rgb * a, a], 1)

## The model

`BacteriaNCA` keeps the manifold of `env_growth.py`: nutrient -> env encoder -> latent
-> DNA decoder -> the update net's weights, as a residual on a learned base rule
(predictor zero-init). Three inputs are new:

- **sense** `(A, R, food)` is concatenated to the state before perception, so every cell
  sees the fields and their Sobel gradients -- it *learns* greedy chemotaxis;
- **kill** = blue or black: `state *= 1 - kill` after every step, so walls and toxins are
  exactly empty by construction;
- **boost** = green: those cells take a second update in the same step -- local time runs
  twice as fast (the teacher's cells mature and age twice as fast).

Updates are **deterministic** (`fire_rate = 1`). The teacher is deterministic, and a stochastic
rule cannot count a cell's age exactly or keep its front in step with the teacher's front -- the
first stochastic model learned to hedge with a faint grey smear where the colony *might* be.

In [13]:
class BacteriaNCA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.env_encoder = nn.Sequential(
            nn.Linear(1, cfg.env_hidden), nn.ReLU(),
            nn.Linear(cfg.env_hidden, cfg.latent_dim))

        self.n_generated = cfg.hidden * cfg.perception + cfg.hidden + cfg.channels * cfg.hidden
        self.dna_decoder = nn.Sequential(nn.Linear(cfg.latent_dim, cfg.dna_hidden), nn.ReLU())
        self.predictor = nn.Linear(cfg.dna_hidden, self.n_generated)
        nn.init.zeros_(self.predictor.weight)          ## start as a zero residual: every
        nn.init.zeros_(self.predictor.bias)            ## nutrient decodes to the base rule

        self.base_w1 = nn.Parameter(torch.randn(cfg.hidden, cfg.perception) * 0.001)
        self.base_b1 = nn.Parameter(torch.zeros(cfg.hidden))
        self.base_w2 = nn.Parameter(torch.zeros(cfg.channels, cfg.hidden))
        self.log_leak = nn.Parameter(torch.tensor(math.log(cfg.leak_init)))

        sx = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]])
        sy = torch.tensor([[1.0, 2.0, 1.0], [0.0, 0.0, 0.0], [-1.0, -2.0, -1.0]])
        c3 = cfg.channels + 3
        self.register_buffer("kx", sx.view(1, 1, 3, 3).repeat(c3, 1, 1, 1))
        self.register_buffer("ky", sy.view(1, 1, 3, 3).repeat(c3, 1, 1, 1))

    def weights_for(self, n):
        """(B,) nutrient levels -> the update net's (w1, b1, w2), batched over B."""
        cfg = self.cfg
        z = self.env_encoder(n.view(-1, 1))
        theta = self.predictor(self.dna_decoder(z))
        b, i, k = z.shape[0], 0, cfg.hidden * cfg.perception
        w1 = self.base_w1 + theta[:, i:i + k].view(b, cfg.hidden, cfg.perception)
        i += k
        b1 = self.base_b1 + theta[:, i:i + cfg.hidden]
        i += cfg.hidden
        w2 = self.base_w2 + theta[:, i:].view(b, cfg.channels, cfg.hidden)
        return w1, b1, w2

    def perceive(self, state, sense):
        x = torch.cat([state, sense], dim=1)               ## the fields ARE sensed
        p = F.pad(x, (1, 1, 1, 1), mode="replicate")       ## no-flux, not a torus
        gx = F.conv2d(p, self.kx, groups=self.cfg.channels + 3)
        gy = F.conv2d(p, self.ky, groups=self.cfg.channels + 3)
        return torch.cat([x, gx, gy], dim=1)

    def alive_mask(self, state):
        return F.max_pool2d(state[:, 3:4], 3, stride=1, padding=1) > self.cfg.alive_threshold

    def _update(self, state, weights, sense):
        w1, b1, w2 = weights
        pre = self.alive_mask(state)
        p = self.perceive(state, sense)
        a = torch.einsum("bop,bpij->boij", w1, p) + b1[:, :, None, None]
        ds = torch.einsum("bcv,bvij->bcij", w2, F.relu(a))
        if self.cfg.fire_rate < 1.0:                       ## optional stochastic updates
            ds = ds * (torch.rand_like(ds[:, :1]) < self.cfg.fire_rate).to(ds.dtype)
        out = state + self.log_leak.exp() * ds
        return out * (pre & self.alive_mask(out)).to(out.dtype)

    def forward(self, state, weights, sense, kill, boost):
        """One step. sense (B,3,H,W); kill, boost (B,1,H,W) in {0,1}. Green (boost) cells take
        a SECOND update in the same step -- local time runs twice as fast there."""
        out = self._update(state, weights, sense)
        out = out + (self._update(out, weights, sense) - out) * boost
        return out * (1.0 - kill)


def seed_state(rgba, cfg):
    """NCA start state: the teacher's first body in RGBA, hidden channels zero."""
    s = torch.zeros(rgba.shape[0], cfg.channels, *rgba.shape[-2:], device=rgba.device)
    s[:, 0:4] = rgba
    return s


class Env:
    """World + fields + teacher, advanced together. The NCA reads `inputs()`.

    On CUDA, `step()` is captured once as a CUDA graph and replayed: the env is ~250 tiny
    kernels per step on a 48x48 grid, so without the graph it is pure launch overhead
    (~12 ms/step on a Windows laptop vs ~0.3 ms of actual GPU work). Every state update
    in World/Fields/Teacher is therefore in place (static addresses)."""

    def __init__(self, cfg, world, use_graph=True):
        self.cfg, self.world = cfg, world
        self.masks = world.masks()
        B, _, H, W = self.masks.shape
        self.fields = Fields(cfg, B, H, W, self.masks.device)
        self.fields.relax(self.masks, cfg.init_iters)
        self.teacher = Teacher(cfg, world.seed, self.masks)
        self.use_graph = use_graph and self.masks.is_cuda
        self._graph, self._init_graph = None, None
        self._n = torch.zeros(B, device=self.masks.device)
        self._sel = torch.zeros(B, 1, 1, 1, dtype=torch.bool, device=self.masks.device)

    def _state_tensors(self):
        w, t = self.world, self.teacher
        return [w.pos, w.vel, w.active, self.masks, self.fields.c, t.age, t.alive, t.t]

    def _step_impl(self):
        self.world.step()
        self.masks.copy_(self.world.masks())
        self.fields.relax(self.masks, self.cfg.field_iters)
        self.teacher.step(self.fields, self.masks, self._n)

    _INIT_CHUNK = 100

    def _init_impl(self):
        self.fields.relax(self.masks, self._INIT_CHUNK, sel=self._sel)

    def _capture(self, fn):
        saved = [x.clone() for x in self._state_tensors()]
        gen_state = self.world.gen.get_state()
        side = torch.cuda.Stream()
        side.wait_stream(torch.cuda.current_stream())
        with torch.cuda.stream(side):                    ## warm-up, as CUDA graphs require
            for _ in range(3):
                fn()
        torch.cuda.current_stream().wait_stream(side)
        for x, v in zip(self._state_tensors(), saved):   ## undo the warm-up steps
            x.copy_(v)
        self.world.gen.set_state(gen_state)
        g = torch.cuda.CUDAGraph()
        g.register_generator_state(self.world.gen)
        with torch.cuda.graph(g):
            fn()
        return g

    def reset_slots(self, idx, rng):
        """Fresh random worlds (and teachers, fields) in slots idx. The fields of just those
        slots are relaxed from zero, `init_iters` steps, by a replayed CUDA graph."""
        idx = torch.as_tensor(idx, dtype=torch.long, device=self.masks.device)
        self.world.splice(idx, World.random(self.cfg, len(idx), rng, self.masks.device))
        self.masks.copy_(self.world.masks())
        self.fields.c[idx] = 0.0
        self._sel.zero_()
        self._sel[idx] = True
        if self.use_graph:
            if self._init_graph is None:
                self._init_graph = self._capture(self._init_impl)
            for _ in range(max(1, self.cfg.init_iters // self._INIT_CHUNK)):
                self._init_graph.replay()
        else:
            self.fields.relax(self.masks, self.cfg.init_iters, sel=self._sel)
        self.teacher.splice(idx, Teacher(self.cfg, self.world.seed[idx], self.masks[idx]))

    def step(self, n):
        self._n.copy_(torch.as_tensor(n, dtype=torch.float32, device=self._n.device).expand_as(self._n))
        if not self.use_graph:
            return self._step_impl()
        if self._graph is None:
            self._graph = self._capture(self._step_impl)
        self._graph.replay()

    def inputs(self):
        m = self.masks
        kill = torch.maximum(m[:, BLUE:BLUE + 1], m[:, BLACK:BLACK + 1])
        return self.fields.sense(m), kill, m[:, GREEN:GREEN + 1]


def _as_n(v, B, device):
    return torch.as_tensor(v, dtype=torch.float32, device=device).expand(B).contiguous()


@torch.no_grad()
def run_world(model, env, n_fn, steps, state=None, record_every=1, seed=0):
    """Advance env + NCA together under nutrient n_fn(t) (float or (B,) tensor).
    Returns the final NCA state and frames [(teacher_rgba, nca_rgba, masks)] on the CPU,
    starting with the initial frame."""
    torch.manual_seed(seed)
    dev = env.masks.device
    B = env.masks.shape[0]
    if state is None:
        state = seed_state(env.teacher.rgba(env.masks), model.cfg)
    snap = lambda: (env.teacher.rgba(env.masks).cpu(), state[:, :4].clamp(0, 1).cpu(), env.masks.cpu())
    frames, cur, w = [snap()], None, None
    for t in range(steps):
        n = _as_n(n_fn(t), B, dev)
        if cur is None or not torch.equal(n, cur):
            w, cur = model.weights_for(n), n
        env.step(n)
        state = model(state, w, *env.inputs())
        if (t + 1) % record_every == 0:
            frames.append(snap())
    return state, frames

## Rendering helpers

`composite()` draws a colony's RGBA over the scene: checkerboard agar, the people as tinted
discs in their clothing colour (blue walls solid), and the colony on top.

In [14]:
PERSON_RGB = np.array([[0.86, 0.16, 0.16],     ## red: food
                       [0.96, 0.58, 0.12],     ## orange: decoy
                       [0.10, 0.10, 0.12],     ## black: toxin
                       [0.20, 0.34, 0.86],     ## blue: wall
                       [0.22, 0.70, 0.30]])    ## green: booster


def _checkerboard(h, w, tile=2):
    yy, xx = np.mgrid[0:h, 0:w]
    return np.where(((yy // tile) + (xx // tile)) % 2 == 0, 0.74, 0.82)


def composite(rgba, masks=None, bg = None):
    """(4,H,W) colony RGBA (+ optional (5,H,W) masks) -> (H,W,3) image in [0,1]."""
    rgba = torch.as_tensor(rgba).detach().float().clamp(0, 1).cpu().numpy()
    rgb, a = rgba[0:3].transpose(1, 2, 0), rgba[3][..., None]
    h, w = rgb.shape[:2]
    if bg is None:
        bg = _checkerboard(h, w)[..., None] * np.ones(3)
    elif bg.shape[:2] != (h,w):
        bg = cv2.resize(bg, (h,w), interpolation = cv2.INTER_AREA)
    if masks is not None:
        m = torch.as_tensor(masks).cpu().numpy() > 0
        for c in (GREEN, ORANGE, RED, BLACK):          ## people: tinted, colony shows through
            bg[m[c]] = 0.35 * bg[m[c]] + 0.65 * PERSON_RGB[c]
        bg[m[BLUE]] = PERSON_RGB[BLUE]
    return np.clip(rgb * a + bg * (1 - a), 0, 1)


def upscale(a, s=5):
    return np.kron(a, np.ones((s, s, 1)))


def frame_with_bar(img, level, bar_h=8):
    """Append a nutrient-level bar below an (H,W,3) image."""
    h, w, _ = img.shape
    bar = np.full((bar_h + 4, w, 3), 0.92)
    bar[2:-2, 2:2 + int(round(level * (w - 4)))] = np.array([0.30, 0.62, 0.32])
    return np.concatenate([img, bar], axis=0)


def side_by_side(*imgs, pad=4):
    h = imgs[0].shape[0]
    sep = np.ones((h, pad, 3))
    out = [imgs[0]]
    for im in imgs[1:]:
        out += [sep, im]
    return np.concatenate(out, axis=1)


def save_gif(frames_rgb, path, duration=60):
    imgs = [Image.fromarray((np.clip(f, 0, 1) * 255).astype(np.uint8)) for f in frames_rgb]
    imgs[0].save(path, save_all=True, append_images=imgs[1:], duration=duration, loop=0)
    print(f"saved {path} ({len(imgs)} frames)")


def save_grid_png(images, ncols, path, labels=None, scale=5, pad=4):
    """Tile (H,W,3) arrays row-major into one PNG."""
    tiles = [upscale(im, scale) for im in images]
    th, tw, _ = tiles[0].shape
    nrows = math.ceil(len(tiles) / ncols)
    canvas = np.ones((nrows * (th + pad) + pad, ncols * (tw + pad) + pad, 3))
    for i, t in enumerate(tiles):
        r, c = divmod(i, ncols)
        y, x = pad + r * (th + pad), pad + c * (tw + pad)
        canvas[y:y + th, x:x + tw] = t
    Image.fromarray((canvas * 255).astype(np.uint8)).save(path)
    print(f"saved {path}" + (f"  [{' | '.join(labels)}]" if labels else ""))

In [9]:
## the data itself, before any training: 4 random moving worlds, the teacher's colony
## every 60 steps at a mid nutrient. If these don't look right, nothing downstream will.
_env = Env(CFG, World.random(CFG, 4, np.random.default_rng(7)))
_imgs = [[] for _ in range(4)]
for t in range(241):
    if t % 60 == 0:
        for b in range(4):
            _imgs[b].append(composite(_env.teacher.rgba(_env.masks)[b], _env.masks[b]))
    _env.step(torch.full((4,), 0.6, device=DEVICE))
save_grid_png([im for row in _imgs for im in row], 5, os.path.join(CFG.out_dir, "bact_live_data.png"),
              labels=["rows: 4 random worlds | cols: teacher colony at t = 0, 60, ..., 240 (n = 0.6)"])
del _env, _imgs

saved outputs\bact_live_data.png  [rows: 4 random worlds | cols: teacher colony at t = 0, 60, ..., 240 (n = 0.6)]


## Sanity checks

Unit tests for every component above, run before any training time is spent. `run_tests()` runs every `test_*` function (optionally filtered by a substring) and reports each one; nothing is skipped silently.

In [15]:
DEV, G = DEVICE, CFG.grid


def disc(y, x, r, g=G):
    yy, xx = np.mgrid[0:g, 0:g]
    return (yy - y) ** 2 + (xx - x) ** 2 <= r * r


def masks_from(red=None, orange=None, black=None, blue=None, green=None, g=G):
    """(1,5,g,g) mask tensor from optional (g,g) bool arrays."""
    m = np.zeros((1, 5, g, g), np.float32)
    for i, a in enumerate((red, orange, black, blue, green)):
        if a is not None:
            m[0, i] = a
    return torch.from_numpy(m).to(DEV)


def relaxed(masks, iters=3000):
    f = Fields(CFG, 1, G, G, DEV)
    f.relax(masks, iters)
    return f

In [16]:
## ---- fields

def test_fields_open_space():
    red = disc(24, 24, 3)
    f = relaxed(masks_from(red=red))
    A = f.c[0, 0].cpu().numpy()
    assert np.allclose(A[red], 1.0), "attractant not pinned to 1 on the red source"
    row = A[24, 27:46]                          ## walking away from the disc edge
    assert np.all(np.diff(row) < 0), f"A not monotone with distance: {row}"
    assert 0.15 <= A[24, 37] <= 0.6, f"A ten cells from the source edge = {A[24, 37]:.3f}"


def test_fields_orange_weaker_and_walls_shade():
    orange = disc(24, 10, 3)
    wall = np.zeros((G, G), bool)
    wall[:, 20:23] = True
    A = relaxed(masks_from(orange=orange, blue=wall)).c[0, 0].cpu().numpy()
    A_open = relaxed(masks_from(orange=orange)).c[0, 0].cpu().numpy()
    assert np.allclose(A[orange], 0.5), "decoy must pin at 0.5"
    assert 0 < A[24, 26] < 0.6 * A_open[24, 26], \
        f"a solid wall must shade (but not block) the chemical: {A[24, 26]:.4f} vs {A_open[24, 26]:.4f}"


def u_cup():
    """Cup opening LEFT toward x=0, back wall at x=30..32, food behind it at x=40."""
    wall = np.zeros((G, G), bool)
    wall[12:37, 30:33] = True                   ## back
    wall[12:15, 18:33] = True                   ## top arm
    wall[34:37, 18:33] = True                   ## bottom arm
    return wall


def test_fields_u_cup_is_a_trap():
    """The point of wall seepage: inside the cup, right at the back wall, the utility is
    a LOCAL MAXIMUM -- greedy ascent from the open side gets stuck there."""
    wall = u_cup()
    f = relaxed(masks_from(red=disc(24, 40, 3), blue=wall))
    A = f.c[0, 0].cpu().numpy()
    inner = A[16:33, 19:30]                     ## the cup's free interior
    y, x = np.unravel_index(inner.argmax(), inner.shape)
    y, x = y + 16, x + 19
    nb = A[y - 1:y + 2, x - 1:x + 2].copy()
    nb[wall[y - 1:y + 2, x - 1:x + 2]] = -1
    assert A[y, x] >= nb.max() - 1e-9, "no local maximum in the cup"
    assert x == 29, f"the trap should sit against the back wall, got x={x}"
    assert A[24, 5] < A[y, x], "the cup must be uphill from the open side"


def test_fields_slit_is_weaker_than_open():
    red = disc(24, 10, 3)
    wall = np.zeros((G, G), bool)
    wall[:, 20:23] = True
    wall[22:27, 20:23] = False                  ## a 5-cell slit
    A_slit = relaxed(masks_from(red=red, blue=wall)).c[0, 0].cpu().numpy()
    A_open = relaxed(masks_from(red=red)).c[0, 0].cpu().numpy()
    behind = A_slit[10, 30] / A_open[10, 30]
    through = A_slit[24, 30] / A_open[24, 30]
    assert behind < 0.8, f"the wall does not shade the field ({behind:.2f})"
    assert through > behind, f"the slit should pass more than the wall ({through:.2f} vs {behind:.2f})"


def test_fields_repellent_shorter_range():
    black = disc(24, 24, 3)
    f = relaxed(masks_from(red=disc(24, 24, 3), black=black))
    A, R = f.c[0, 0].cpu().numpy(), f.c[0, 1].cpu().numpy()
    assert np.allclose(R[black], 1.0)
    assert R[24, 35] < A[24, 35], "repellent should decay faster than attractant"


def test_fields_lag_moving_source():
    """Warm-started relaxation: a moved source is followed, but not instantly far away."""
    f = relaxed(masks_from(red=disc(24, 10, 3)))
    far_before = f.c[0, 0, 24, 44].item()
    m2 = masks_from(red=disc(24, 30, 3))
    f.relax(m2, CFG.field_iters)
    lagged = f.c[0, 0, 24, 44].item()
    f.relax(m2, 3000)
    settled = f.c[0, 0, 24, 44].item()
    assert far_before < lagged < settled, (far_before, lagged, settled)


def test_fields_sense():
    red = disc(24, 24, 3)
    m = masks_from(red=red)
    f = relaxed(m)
    s = f.sense(m)
    assert s.shape == (1, 3, G, G)
    assert torch.equal(s[:, 2], m[:, 0]), "third sense channel must be the raw food mask"
    assert torch.allclose(s[0, 0][torch.from_numpy(red).to(DEV)], torch.ones(1, device=DEV), atol=1e-4)
    assert (s[:, 0] >= 0).all() and (s[:, 0] <= 1).all()

In [12]:
## ---- world

def test_world_masks_binary_and_discs():
    w = World.scripted(CFG, [[dict(pos=(20, 20), vel=(0, 0), r=3, colour="orange"),
                                 dict(pos=(30, 35), vel=(0, 0), r=2, colour="green")]],
                          seeds=[(5, 5)], device=DEV)
    m = w.masks()
    assert m.shape == (1, 5, G, G)
    assert set(torch.unique(m).tolist()) <= {0.0, 1.0}
    assert np.array_equal(m[0, ORANGE].cpu().numpy() > 0, disc(20, 20, 3))
    assert np.array_equal(m[0, GREEN].cpu().numpy() > 0, disc(30, 35, 2))
    assert m[0, RED].sum() == 0 and m[0, BLUE].sum() == 0


def test_world_scripted_motion_and_bounce():
    w = World.scripted(CFG, [[dict(pos=(24, 24), vel=(0.0, 0.25), r=2, colour="red")]],
                          seeds=[(5, 5)], device=DEV)
    for _ in range(8):
        w.step()
    assert torch.allclose(w.pos[0, 0], torch.tensor([24.0, 26.0], device=DEV)), w.pos[0, 0]
    for _ in range(400):                        ## long enough to hit a border many times
        w.step()
        x = w.pos[0, 0, 1].item()
        assert 2 - 1e-5 <= x <= G - 3 + 1e-5, f"person left the grid: x={x}"


def test_world_static_walls_are_blue():
    walls = np.zeros((1, G, G), bool)
    walls[0, 10:12, :] = True
    w = World.scripted(CFG, [[]], seeds=[(30, 30)], walls=walls, device=DEV)
    assert np.array_equal(w.masks()[0, BLUE].cpu().numpy() > 0, walls[0])


def test_world_random_seed_valid_and_deterministic():
    w1 = World.random(CFG, 16, np.random.default_rng(3), DEV)
    w2 = World.random(CFG, 16, np.random.default_rng(3), DEV)
    m = w1.masks()
    for b in range(16):
        y, x = w1.seed[b].tolist()
        assert m[b, BLUE, y, x] == 0 and m[b, BLACK, y, x] == 0, "seed on blue/black"
    for _ in range(50):
        w1.step(); w2.step()
    assert torch.equal(w1.masks(), w2.masks()), "same rng must give the same world"
    assert (m[:, RED].flatten(1).sum(1) > 0).float().mean() >= 0.5, "red too rare"

In [17]:
## ---- teacher

def dilate_np(mask):
    return F.max_pool2d(torch.from_numpy(mask).float()[None, None], 3, stride=1, padding=1)[0, 0].numpy() > 0


def run_teacher(masks, seed, n, steps, masks_fn=None):
    """Static (or scripted via masks_fn(t)) world; returns the teacher and the track of its
    front (the rightmost living cell -- every scene below travels rightward)."""
    f = relaxed(masks)
    T = Teacher(CFG, torch.tensor([seed], device=DEV), masks)
    track = [front(T)]
    for t in range(steps):
        m = masks if masks_fn is None else masks_fn(t)
        f.relax(m, CFG.field_iters)
        T.step(f, m, torch.full((1,), float(n), device=DEV))
        track.append(front(T))
    return T, track


def front(T):
    ys, xs = np.nonzero(torch.isfinite(T.age[0]).cpu().numpy())
    if len(xs) == 0:
        return None
    i = np.argmax(xs)
    return int(ys[i]), int(xs[i])


def test_teacher_climbs_to_food():
    red = disc(24, 40, 3)
    T, track = run_teacher(masks_from(red=red), (24, 8), 0.5, 140)
    alive = torch.isfinite(T.age[0]).cpu().numpy()
    assert alive[red].all(), f"colony should engulf the food, has {alive[red].sum()}/{red.sum()}"
    assert track[-1] == track[-10], "the front should rest once on the food"


def test_teacher_trapped_in_u_cup():
    T, track = run_teacher(masks_from(red=disc(24, 40, 3), blue=u_cup()), (24, 8), 1.0, 250)
    ys, xs = np.nonzero(torch.isfinite(T.age[0]).cpu().numpy())
    assert len(xs) > 0, "the trapped colony died"
    assert xs.max() == 29 and xs.min() >= 19 and 15 <= ys.min() and ys.max() <= 33, \
        f"greedy colony should sit at the cup's back wall, spans x {xs.min()}..{xs.max()}"
    assert track[-1] == track[-30], "a trapped colony must not move on"


def test_teacher_biomass_and_retreat():
    m = masks_from(red=disc(24, 44, 2))
    counts = {}
    for n in (0.25, 1.0):
        T, _ = run_teacher(m, (24, 4), n, 60)
        alive = torch.isfinite(T.age)
        counts[n] = alive.sum().item()
        assert T.age[alive].max().item() <= life(n, CFG), "a cell outlived life(n)"
    assert counts[1.0] > 1.5 * counts[0.25], counts
    ## drop the nutrient: the tail dies on the very next step
    f = relaxed(m)
    T.step(f, m, torch.full((1,), 0.25, device=DEV))
    assert T.age[torch.isfinite(T.age)].max().item() <= life(0.25, CFG)


def test_teacher_kills_under_black_and_blue():
    m = masks_from(red=disc(24, 44, 2))
    T, _ = run_teacher(m, (24, 4), 1.0, 60)
    body = torch.isfinite(T.age[0]).cpu().numpy()
    ys, xs = np.nonzero(body)
    y0, x0 = int(ys.mean()), int(xs.min()) + 2     ## the old tail
    tox = disc(y0, x0, 2)
    wall = np.zeros((G, G), bool)
    wall[:, 30:32] = True
    m2 = masks_from(red=disc(24, 44, 2), black=tox, blue=wall)
    f = relaxed(m2, 50)
    T.step(f, m2, torch.full((1,), 1.0, device=DEV))
    alive = torch.isfinite(T.age[0]).cpu().numpy()
    assert not alive[tox].any(), "cells survived under black"
    assert not alive[wall].any(), "cells survived under blue"
    assert alive.any()


def test_teacher_green_doubles_speed():
    red = disc(24, 46, 1)
    green = np.zeros((G, G), bool)
    green[18:31, :] = True                       ## a green carpet along the path
    _, slow = run_teacher(masks_from(red=red), (24, 4), 0.5, 20)
    _, fast = run_teacher(masks_from(red=red, green=green), (24, 4), 0.5, 20)
    d_slow, d_fast = slow[-1][1] - slow[0][1], fast[-1][1] - fast[0][1]
    assert 8 <= d_slow <= 11 and d_fast >= 1.8 * d_slow, f"front displacement slow {d_slow}, fast {d_fast}"


def test_teacher_extinct_when_crushed():
    m = masks_from(red=disc(24, 44, 2))
    T, _ = run_teacher(m, (24, 4), 0.5, 10)
    everything = np.ones((G, G), bool)
    m2 = masks_from(blue=everything)
    f = relaxed(m2, 10)
    T.step(f, m2, torch.full((1,), 0.5, device=DEV))
    assert not T.alive[0].item(), "teacher should be extinct"
    assert (T.rgba(m2)[0, 3] == 0).all()


def test_teacher_feeds_on_food():
    """Reaching food, the colony engulfs the whole food disc and keeps it alive (no aging)."""
    red = disc(24, 30, 3)
    m = masks_from(red=red)
    T, _ = run_teacher(m, (24, 8), 0.5, 60 + 3 * CFG.life_max)
    alive = torch.isfinite(T.age[0]).cpu().numpy()
    assert alive[red].all(), f"only {alive[red].sum()}/{red.sum()} food cells colonised"
    assert (T.age[0][torch.from_numpy(red).to(DEV)] == 0).all(), "feeding cells must not age"


def test_teacher_anchor_round_trip():
    """Anchoring a teacher on its OWN (thick) image recovers its body and ages. Eroding the
    dilation is a morphological closing: every cell comes back, plus a few filled corners."""
    m = masks_from(red=disc(24, 44, 2))
    T, _ = run_teacher(m, (24, 4), 1.0, 40)
    age0 = T.age.clone()
    T.anchor(T.rgba(m), m, torch.ones(1, dtype=torch.bool, device=DEV))
    alive, back = torch.isfinite(age0), torch.isfinite(T.age)
    assert back[alive].all(), "anchoring lost colony cells"
    assert back.sum() <= alive.sum() * 1.1 + 2, f"{back.sum().item()} cells back from {alive.sum().item()}"
    assert (T.age[alive] - age0[alive]).abs().max().item() <= 2, "ages drifted by more than two steps"
    before = T.age.clone()
    T.anchor(torch.zeros_like(T.rgba(m)), m, torch.zeros(1, dtype=torch.bool, device=DEV))
    assert torch.equal(T.age, before), "sel=False must leave the slot alone"


def test_teacher_rgba_colours():
    m = masks_from(red=disc(24, 20, 3))
    T, _ = run_teacher(m, (24, 8), 1.0, 60)
    rgba = T.rgba(m)[0].cpu().numpy()
    alive = torch.isfinite(T.age[0]).cpu().numpy()
    thick = dilate_np(alive)
    assert np.array_equal(rgba[3] > 0, thick), "alpha must be the body dilated by one cell"
    on_food = thick & disc(24, 20, 3)
    assert on_food.any() and np.allclose(rgba[0:3, on_food].T, BLOOM), "no bloom on the food"

In [18]:
## ---- model

def probe_model():
    """A model with a small non-zero predictor (at init every n decodes to the base rule,
    and the base rule outputs zero -- so sensitivities are legitimately 0 there)."""
    m = BacteriaNCA(CFG).to(DEV)
    torch.nn.init.normal_(m.predictor.weight, std=1e-3)
    return m


def random_env(B=4, seed=0):
    return Env(CFG, World.random(CFG, B, np.random.default_rng(seed), DEV))


def test_model_shapes_and_kill_exact():
    m, env = probe_model(), random_env()
    n = torch.full((4,), 0.7, device=DEV)
    w1, b1, w2 = m.weights_for(n)
    assert w1.shape == (4, CFG.hidden, CFG.perception)
    assert b1.shape == (4, CFG.hidden) and w2.shape == (4, CFG.channels, CFG.hidden)
    state = seed_state(env.teacher.rgba(env.masks), CFG)
    for _ in range(20):
        env.step(n)
        state = m(state, (w1, b1, w2), *env.inputs())
    kill = env.inputs()[1][:, 0] > 0
    assert state.abs().amax(1)[kill].max().item() == 0.0, "state non-zero under blue/black"


def test_model_senses_fields():
    m, env = probe_model(), random_env()
    n = torch.full((4,), 0.7, device=DEV)
    w = m.weights_for(n)
    state = seed_state(env.teacher.rgba(env.masks), CFG)
    sense, kill, boost = env.inputs()
    with torch.no_grad():
        torch.manual_seed(1); a = m(state, w, sense, kill, boost)
        torch.manual_seed(1); b = m(state, w, sense.roll(5, -1), kill, boost)
    assert (a - b).abs().max().item() > 0, "rolling the sensed fields changed nothing"


def test_model_grad_reaches_predictor_at_init():
    m, env = BacteriaNCA(CFG).to(DEV), random_env()
    n = torch.full((4,), 0.5, device=DEV)
    state = seed_state(env.teacher.rgba(env.masks), CFG)
    for _ in range(8):
        env.step(n)
        state = m(state, m.weights_for(n), *env.inputs())
    loss = torch.nn.functional.mse_loss(state[:, :4], env.teacher.rgba(env.masks))
    g = torch.autograd.grad(loss, m.predictor.weight)[0].norm().item()
    assert math.isfinite(g) and g > 0, g


def test_model_green_is_a_double_step_and_deterministic():
    m = probe_model()
    with torch.no_grad():
        m.base_w2.normal_(std=0.1)              ## make every cell's update non-zero
    s = torch.zeros(1, CFG.channels, G, G, device=DEV)
    s[:, 3] = 1.0
    sense = torch.rand(1, 3, G, G, device=DEV)
    zero = torch.zeros(1, 1, G, G, device=DEV)
    w = m.weights_for(torch.tensor([0.5], device=DEV))
    with torch.no_grad():
        once, again = m(s, w, sense, zero, zero), m(s, w, sense, zero, zero)
        twice = m(s, w, sense, zero, torch.ones_like(zero))
        ref = m._update(m._update(s, w, sense), w, sense)
    assert torch.equal(once, again), "updates must be deterministic"
    assert torch.allclose(twice, ref, atol=1e-6), "green must be exactly two updates"
    assert not torch.allclose(once, twice), "green changed nothing"


def test_run_world_records_frames():
    m, env = probe_model(), random_env(B=2)
    final, frames = run_world(m, env, lambda t: 0.6, 12, record_every=4)
    assert final.shape == (2, CFG.channels, G, G) and len(frames) == 4
    t_rgba, n_rgba, masks = frames[-1]
    assert t_rgba.shape == n_rgba.shape == (2, 4, G, G) and masks.shape == (2, 5, G, G)


def test_env_cuda_graph_matches_eager():
    if not torch.cuda.is_available():
        return
    envs = [Env(CFG, World.random(CFG, 4, np.random.default_rng(5), DEV), use_graph=g)
            for g in (False, True)]
    n = torch.tensor([0.3, 0.6, 0.9, 1.0], device=DEV)
    for t in range(40):
        for e in envs:
            e.step(n)
        if t == 20:                             ## a slot reset mid-run must work with the graph
            for e in envs:
                e.reset_slots(np.array([1, 2]), np.random.default_rng(9))
    a, b = envs
    assert torch.equal(a.masks, b.masks), "masks diverged"
    assert torch.allclose(a.fields.c, b.fields.c, atol=1e-6), "fields diverged"
    assert torch.equal(a.teacher.age, b.teacher.age) and torch.equal(a.teacher.alive, b.teacher.alive)

In [15]:
def run_tests(pat=""):
    tests = [(k, v) for k, v in globals().items() if k.startswith("test_") and callable(v) and pat in k]
    failed = []
    for name, fn in tests:
        t0 = time.time()
        try:
            fn()
            print(f"PASS {name} ({time.time() - t0:.1f}s)")
        except Exception as e:                  ## report every failure, keep going
            failed.append(name)
            print(f"FAIL {name}: {type(e).__name__}: {e}")
    print(f"{len(tests) - len(failed)}/{len(tests)} passed")
    return failed


assert not run_tests(), "sanity checks failed"

PASS test_fields_open_space (1.0s)
PASS test_fields_orange_weaker_and_walls_shade (1.9s)
PASS test_fields_u_cup_is_a_trap (0.7s)
PASS test_fields_slit_is_weaker_than_open (1.6s)
PASS test_fields_repellent_shorter_range (0.7s)
PASS test_fields_lag_moving_source (1.4s)
PASS test_fields_sense (0.6s)
PASS test_world_masks_binary_and_discs (0.0s)
PASS test_world_scripted_motion_and_bounce (0.0s)
PASS test_world_static_walls_are_blue (0.0s)
PASS test_world_random_seed_valid_and_deterministic (0.0s)
PASS test_teacher_climbs_to_food (1.5s)
PASS test_teacher_trapped_in_u_cup (2.0s)
PASS test_teacher_biomass_and_retreat (2.6s)
PASS test_teacher_kills_under_black_and_blue (1.1s)
PASS test_teacher_green_doubles_speed (1.7s)
PASS test_teacher_extinct_when_crushed (0.7s)
PASS test_teacher_feeds_on_food (1.9s)
PASS test_teacher_anchor_round_trip (0.9s)
PASS test_teacher_rgba_colours (0.9s)
PASS test_model_shapes_and_kill_exact (0.9s)
PASS test_model_senses_fields (0.5s)
FAIL test_model_grad_reaches_p

AssertionError: sanity checks failed

## Checkpoint save / load

In [19]:
def save_checkpoint(model, cfg, path=None):
    path = path or os.path.join(cfg.out_dir, cfg.ckpt_name)
    torch.save({"state_dict": model.state_dict(), "config": asdict(cfg)}, path)
    print(f"saved {path}")


def load_checkpoint(path=None, device=DEVICE):
    path = path or os.path.join(CFG.out_dir, CFG.ckpt_name)
    ck = torch.load(path, map_location=device, weights_only=False)
    known = Config.__dataclass_fields__             ## tolerate configs from older runs
    cfg = Config(**{k: v for k, v in ck["config"].items() if k in known})
    m = BacteriaNCA(cfg).to(device)
    m.load_state_dict(ck["state_dict"])
    m.eval()
    return m, cfg

## Training -- imitate the teacher from the NCA's own states

A **persistent pool** of `batch` running worlds. The teacher and the NCA start from the
same body and run in lockstep on identical fields; each epoch continues every world by
a short **gradient window** (MSE on RGBA vs the teacher every `loss_every` steps), then
detaches and carries the state on. Slots are reset to fresh worlds at random -- often
early in training (short episodes), rarely later (episodes hundreds of steps deep) --
so the rule is trained to *correct back* toward the teacher from wherever it has
drifted, which is what keeps an unbounded live run stable. Each slot has its own
nutrient schedule: constant, a ramp, or a sudden jump, up or down.

In [18]:
class Schedule:
    """Per-slot nutrient schedules on per-slot clocks: constant / ramp / jump."""

    def __init__(self, B, rng, cfg, device=DEVICE):
        self.cfg, self.device = cfg, device
        self.a, self.b = np.zeros(B), np.zeros(B)
        self.kind, self.t_s, self.dur, self.t = (np.zeros(B, int) for _ in range(4))
        self.reset(np.arange(B), rng)

    def reset(self, idx, rng):
        k = len(idx)
        self.a[idx] = rng.uniform(self.cfg.n_lo, 1.0, k)
        self.b[idx] = rng.uniform(self.cfg.n_lo, 1.0, k)
        self.kind[idx] = rng.integers(0, 3, k)
        self.t_s[idx] = rng.integers(0, 400, k)
        self.dur[idx] = rng.integers(40, 200, k)
        self.t[idx] = 0

    def n(self):
        f = np.clip((self.t - self.t_s) / self.dur, 0.0, 1.0)
        v = np.where(self.kind == 0, self.a,
                     np.where(self.kind == 1, self.a + (self.b - self.a) * f,
                              np.where(self.t >= self.t_s, self.b, self.a)))
        return torch.as_tensor(v, dtype=torch.float32, device=self.device)

    def tick(self):
        self.t += 1


def _fixed_eval_worlds(cfg, device=DEVICE):
    return World.random(cfg, 4, np.random.default_rng(1234), device)


@torch.no_grad()
def _fixed_eval(model, cfg):
    """Deterministic held-out eval for best-checkpoint tracking: MSE vs the teacher at
    t = 100, 200, 300 on four fixed worlds. Saves and restores the global RNG."""
    cpu_state = torch.get_rng_state()
    cuda_states = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    try:
        dev = next(model.parameters()).device
        env = Env(cfg, _fixed_eval_worlds(cfg, dev))
        levels = torch.tensor([0.4, 0.7, 1.0, 0.6], device=dev)
        _, frames = run_world(model, env, lambda t: levels, 300, record_every=100, seed=11)
        return float(np.mean([F.mse_loss(n, t).item() for t, n, _ in frames[1:]]))
    finally:
        torch.set_rng_state(cpu_state)
        if cuda_states is not None:
            torch.cuda.set_rng_state_all(cuda_states)


def reset_prob(epoch, epochs):
    """Per-slot reset probability per epoch: 0.3 early (episodes ~80 steps) decaying to
    0.05 by mid-training (episodes ~500 steps)."""
    f = min(1.0, epoch / (0.5 * epochs))
    return 0.3 + (0.05 - 0.3) * f


def train(model, cfg, rng, epochs=None):
    dev = next(model.parameters()).device
    epochs = epochs or cfg.epochs
    vec = [p for p in model.parameters() if p.numel() > 1]
    sca = [p for p in model.parameters() if p.numel() == 1]
    opt = torch.optim.Adam([{"params": vec, "lr": cfg.lr},
                            {"params": sca, "lr": cfg.scalar_lr}])
    best, best_state = float("inf"), copy.deepcopy(model.state_dict())
    history, t0 = [], time.time()

    B = cfg.batch
    env = Env(cfg, World.random(cfg, B, rng, dev))
    sched = Schedule(B, rng, cfg, dev)
    state = seed_state(env.teacher.rgba(env.masks), cfg)

    for epoch in range(epochs):
        frac = epoch / max(epochs - 1, 1)
        opt.param_groups[0]["lr"] = cfg.lr * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * frac)))

        ## reset some slots: at random, or because the teacher colony went extinct
        idx = np.nonzero((rng.random(B) < reset_prob(epoch, epochs))
                         | ~env.teacher.alive.cpu().numpy())[0]
        if len(idx):
            env.reset_slots(idx, rng)
            sched.reset(idx, rng)
            state[idx] = seed_state(env.teacher.rgba(env.masks)[idx], cfg)

        ## DAgger: most slots re-anchor the teacher on the NCA's own colony, so the loss asks
        ## "from where YOU are, what would the teacher do next" (no compounding divergence)
        sync = torch.as_tensor(rng.random(B) < cfg.dagger_prob, device=dev)
        env.teacher.anchor(state[:, 0:4].detach().clamp(0, 1), env.masks, sync)

        opt.zero_grad()
        W = int(rng.integers(cfg.window[0], cfg.window[1] + 1))
        loss, k = 0.0, 0
        for i in range(W):
            n = sched.n()
            env.step(n)
            sched.tick()
            state = model(state, model.weights_for(n), *env.inputs())
            if (i + 1) % cfg.loss_every == 0 or i == W - 1:
                loss = loss + F.mse_loss(state[:, 0:4], env.teacher.rgba(env.masks))
                k += 1
        loss = loss / k
        loss.backward()
        for p in vec:                                   ## per-parameter gradient normalisation
            if p.grad is not None:
                p.grad /= p.grad.norm() + 1e-8
        opt.step()
        state = state.detach()
        history.append((epoch, loss.item(), float(sched.t.mean())))

        if epoch % cfg.eval_every == 0 or epoch == epochs - 1:
            ev = _fixed_eval(model, cfg)
            if ev < best:
                best, best_state = ev, copy.deepcopy(model.state_dict())

        if epoch % cfg.log_every == 0 or epoch == epochs - 1:
            recent = np.mean([h[1] for h in history[-cfg.log_every:]])
            print(f"epoch {epoch:5d} | loss {recent:.5f} | mean episode depth {sched.t.mean():5.0f} "
                  f"| best-eval {best:.5f} | {time.time() - t0:.0f}s", flush=True)

    model.load_state_dict(best_state)
    print(f"done in {time.time() - t0:.1f}s | best eval {best:.6f} | weights restored to best")
    return history

In [20]:
TRAIN = False   ## True: train from scratch (~30 min on an RTX 4060) and save outputs/bacteria_live.pth
                ## False: load that checkpoint

torch.manual_seed(CFG.seed)
if TRAIN:
    model = BacteriaNCA(CFG).to(DEVICE)
    history = train(model, CFG, np.random.default_rng(CFG.seed))
    save_checkpoint(model, CFG)
else:
    model, CFG = load_checkpoint()
model.eval()
print("ready")

ready


## Verification -- the acceptance tests

On worlds the training rng never produced (rng 777/778) and on hand-built scenes with
scripted people. `P(y, x, r, colour, vy, vx)` is one person; the scenes are small enough to read:

| scene | what it proves |
|---|---|
| `chase` | a red person walks away; the colony follows |
| `decoy` / `decoy_only` | red and orange equidistant -> red wins; orange alone still attracts |
| `toxin` | a black person between seed and food -> detour, nothing lives near black |
| `trap` | a U-shaped blue wall with the food behind it -> **stuck in the cup** (greedy, not BFS) |
| `green` / `no_green` | a green carpet along the path -> the front gets further in the same time |
| `open` | biomass grows with `n`; dropping `n` retreats with no ghost |

In [20]:
def P(y, x, r, colour, vy=0.0, vx=0.0):
    return dict(pos=(y, x), vel=(vy, vx), r=r, colour=colour)


def u_cup_walls(g):
    """Cup opening LEFT toward the seed, back wall at x=30..32, food behind it at x=40."""
    wall = np.zeros((g, g), bool)
    wall[12:37, 30:33] = True
    wall[12:15, 18:33] = True
    wall[34:37, 18:33] = True
    return wall


def scenes(cfg):
    g = cfg.grid
    return {
        "chase":      dict(people=[P(24, 30, 3, "red", 0.12, 0.08)], seed=(24, 6)),
        "decoy":      dict(people=[P(24, 42, 3, "red"), P(24, 6, 3, "orange")], seed=(24, 24)),
        "decoy_only": dict(people=[P(24, 6, 3, "orange")], seed=(24, 24)),
        "toxin":      dict(people=[P(24, 42, 3, "red"), P(25, 24, 4, "black")], seed=(24, 6)),
        "trap":       dict(people=[P(24, 40, 3, "red")], seed=(24, 8), walls=u_cup_walls(g)),
        "green":      dict(people=[P(24, 46, 1, "red")] + [P(24, x, 4, "green") for x in (10, 18, 26, 34)],
                           seed=(24, 4)),
        "no_green":   dict(people=[P(24, 46, 1, "red")], seed=(24, 4)),
        "open":       dict(people=[P(24, 44, 2, "red")], seed=(24, 4)),
    }


def scene_env(cfg, name):
    s = scenes(cfg)[name]
    walls = None if s.get("walls") is None else s["walls"][None]
    return Env(cfg, World.scripted(cfg, [s["people"]], [s["seed"]], walls=walls))


def _alpha(rgba):
    return rgba[0, 3].numpy() if rgba.dim() == 4 else rgba[3].numpy()


def disc_mask(g, y, x, r):
    yy, xx = np.mgrid[0:g, 0:g]
    return (yy - y) ** 2 + (xx - x) ** 2 <= r * r


def dilate(mask, r=2):
    """Chebyshev dilation of a bool mask by r cells."""
    out = torch.from_numpy(mask).float()[None, None]
    return (F.max_pool2d(out, 2 * r + 1, stride=1, padding=r)[0, 0] > 0).numpy()


def near(g, person, pad):
    return disc_mask(g, person["pos"][0], person["pos"][1], person["r"] + pad)


def run_scene(model, cfg, name, n, steps, seed=0, record_every=None):
    env = scene_env(cfg, name)
    _, frames = run_world(model, env, n if callable(n) else (lambda t: n), steps,
                          record_every=record_every or steps, seed=seed)
    return env, frames


@torch.no_grad()
def verify(model, cfg=CFG):
    dev, g = next(model.parameters()).device, cfg.grid
    S, results = scenes(cfg), {}

    ## 1. walls and toxins are hard physics: EXACTLY zero at every step, moving ones included
    w = World.random(cfg, 4, np.random.default_rng(777), dev)
    w.col[:, 1], w.col[:, 2] = BLUE, BLACK
    w.active[:, :3] = True
    env = Env(cfg, w)
    state = seed_state(env.teacher.rgba(env.masks), cfg)
    wts = model.weights_for(torch.full((4,), 0.8, device=dev))
    worst = 0.0
    torch.manual_seed(1)
    for _ in range(200):
        env.step(torch.full((4,), 0.8, device=dev))
        sense, kill, boost = env.inputs()
        state = model(state, wts, sense, kill, boost)
        under = state.abs().amax(1, keepdim=True)[kill > 0]
        worst = max(worst, under.max().item() if under.numel() else 0.0)
    results["wall"] = (worst == 0.0, f"max |state| under blue/black over 200 steps: {worst}")

    ## 2. imitation on unseen moving worlds, relative to the empty-grid baseline
    env = Env(cfg, World.random(cfg, 4, np.random.default_rng(778), dev))
    levels = torch.tensor([0.4, 0.7, 1.0, 0.6], device=dev)
    _, frames = run_world(model, env, lambda t: levels, 400, record_every=100, seed=2)
    det, ok = [], True
    for t, (tr, nr, _) in zip((100, 200, 300, 400), frames[1:]):
        mse, base = F.mse_loss(nr, tr).item(), (tr ** 2).mean().item()
        ok &= mse < 0.35 * base
        det.append(f"t={t} {mse:.5f} ({mse / base:.2f}x empty)")
    results["imitation"] = (ok, ", ".join(det) + " (need < 0.35x)")

    ## 3. chase: the colony is still on the walking red person at the end
    env, frames = run_scene(model, cfg, "chase", 0.6, 300, seed=3)
    py, px = env.world.pos[0, 0].tolist()
    ys, xs = np.nonzero(_alpha(frames[-1][1]) > 0.5)
    d = float(np.sqrt((ys - py) ** 2 + (xs - px) ** 2).min()) if len(ys) else float("inf")
    results["chase"] = (d <= 3 + 4, f"colony-to-person distance after 300 steps {d:.1f} (<= 7)")

    ## 4. food beats decoy; a decoy alone still attracts
    _, fr = run_scene(model, cfg, "decoy", 0.6, 200, seed=4)
    a = _alpha(fr[-1][1])
    m_red = a[near(g, S["decoy"]["people"][0], 4)].sum()
    m_or = a[near(g, S["decoy"]["people"][1], 4)].sum()
    _, fr = run_scene(model, cfg, "decoy_only", 0.6, 200, seed=4)
    m_only = _alpha(fr[-1][1])[near(g, S["decoy_only"]["people"][0], 4)].sum()
    results["decoy"] = (m_red > 3 * max(m_or, 1.0) and m_only >= 5,
                        f"alpha mass near red {m_red:.1f} vs orange {m_or:.1f} (> 3x); "
                        f"orange alone {m_only:.1f} (>= 5)")

    ## 5. toxin: detour to the food, nothing under (or hugging) the black person
    _, fr = run_scene(model, cfg, "toxin", 0.6, 200, seed=5, record_every=10)
    m_food = _alpha(fr[-1][1])[near(g, S["toxin"]["people"][0], 4)].sum()
    tox = near(g, S["toxin"]["people"][1], 1)
    m_tox = max(_alpha(f[1])[tox].sum() for f in fr)
    results["toxin"] = (m_food >= 5 and m_tox < 1.0,
                        f"alpha mass at food {m_food:.1f} (>= 5); max alpha within 1 cell of the "
                        f"toxin {m_tox:.2f} (< 1)")

    ## 6. trap: greedy ascent is caught in the cup -- the food behind it is never reached
    _, fr = run_scene(model, cfg, "trap", 1.0, 250, seed=6)
    a, ta = _alpha(fr[-1][1]), _alpha(fr[-1][0])
    food = near(g, S["trap"]["people"][0], 3)
    ys, xs = np.nonzero(a > 0.3)
    in_cup = len(ys) > 0 and 15 <= ys.mean() <= 33 and 19 <= xs.mean() <= 30
    results["trap"] = (a[food].sum() < 1.0 and in_cup and ta[food].sum() < 1.0,
                       f"alpha at food {a[food].sum():.2f} (< 1), colony centroid "
                       f"{(round(ys.mean(), 1), round(xs.mean(), 1)) if len(ys) else None} in the cup: {in_cup}")

    ## 7. biomass grows with n (measured mid-journey, while the trail is still laid down);
    ##    dropping n mid-journey cuts the tail at once -- no ghost left behind
    fr7 = {n: run_scene(model, cfg, "open", n, 60, seed=7)[1][-1] for n in (0.25, 0.5, 1.0)}
    masses = [_alpha(f[1]).sum() for f in fr7.values()]
    t_masses = [_alpha(f[0]).sum() for f in fr7.values()]
    _, fr = run_scene(model, cfg, "open", lambda t: 1.0 if t < 60 else 0.2, 100, seed=8)
    a, ta = _alpha(fr[-1][1]), _alpha(fr[-1][0])
    ghost = a[~dilate(ta > 0, 2)].sum()
    ratio = a.sum() / max(ta.sum(), 1e-6)
    results["biomass"] = (masses[0] < masses[1] < masses[2] and ghost < 5 and 0.5 <= ratio <= 1.5,
                          f"mass at n=0.25/0.5/1: {[round(float(m), 1) for m in masses]} (increasing; "
                          f"teacher {[round(float(m)) for m in t_masses]}); after the drop: ghost "
                          f"{ghost:.1f} (< 5), NCA/teacher mass {ratio:.2f} (0.5-1.5)")

    ## 8. booster: a green carpet takes the front further in the same time
    fx = {}
    for name in ("green", "no_green"):
        xs = np.nonzero(_alpha(run_scene(model, cfg, name, 0.6, 60, seed=9)[1][-1][1]) > 0.5)[1]
        fx[name] = int(xs.max()) if len(xs) else -1
    results["booster"] = (fx["green"] >= fx["no_green"] + 4,
                          f"front x after 60 steps: green {fx['green']} vs plain {fx['no_green']} (+4 needed)")

    print("\n=== verification (held-out worlds + scripted scenes) ===")
    ok = True
    for name, (passed, msg) in results.items():
        print(f"  [{'PASS' if passed else 'FAIL'}] {name:<9} {msg}")
        ok &= bool(passed)
    print(f"=== {'ALL PASSED' if ok else 'FAILURES PRESENT'} ===\n")
    return ok, results


ok, results = verify(model, CFG)


=== verification (held-out worlds + scripted scenes) ===
  [PASS] wall      max |state| under blue/black over 200 steps: 0.0
  [FAIL] imitation t=100 0.04084 (0.76x empty), t=200 0.06743 (0.74x empty), t=300 0.07034 (0.95x empty), t=400 0.08784 (0.64x empty) (need < 0.35x)
  [PASS] chase     colony-to-person distance after 300 steps 0.0 (<= 7)
  [PASS] decoy     alpha mass near red 100.5 vs orange 0.0 (> 3x); orange alone 106.0 (>= 5)
  [PASS] toxin     alpha mass at food 114.9 (>= 5); max alpha within 1 cell of the toxin 0.48 (< 1)
  [PASS] trap      alpha at food 0.00 (< 1), colony centroid (np.float64(24.1), np.float64(24.9)) in the cup: True
  [FAIL] biomass   mass at n=0.25/0.5/1: [23.7, 74.9, 126.5] (increasing; teacher [27, 45, 81]); after the drop: ghost 21.4 (< 5), NCA/teacher mass 0.91 (0.5-1.5)
  [PASS] booster   front x after 60 steps: green 38 vs plain 24 (+4 needed)
=== FAILURES PRESENT ===



**Status of this checkpoint (v5, 4000 epochs, ~32 min on an RTX 4060): 6 of 8 pass.**

- Passing: walls/toxins exactly empty; chases a walking food person; food beats decoy and a lone
  decoy still attracts; detours around a toxin (it splits into two arms and rejoins on the food);
  **trapped in the U-cup** (greedy, not BFS); green speeds the front up.
- `biomass` is half-met: colony mass rises with `n` (and tracks the teacher's), but after `n`
  drops mid-journey old trail cells linger (~21 cells of ghost vs a limit of 5) -- abandoned arms
  fade more slowly than the teacher's.
- `imitation` (long-horizon MSE vs an *independently* running teacher) sits at 0.37-0.78x the
  empty-grid error vs a 0.35x target. Behaviour matches; exact cell-for-cell timing drifts.

What was tried, in order: stochastic rule + single-head teacher (5/8, diffuse fan) -> + DAgger
+ feeding (5/8) -> deterministic updates (5/8) -> fully local teacher with a 1-cell trail (5/8,
too thin to hold) -> the same teacher drawn 3 cells thick (this one, 6/8). Details in the design doc.

## Figures -- teacher (left) vs NCA (right)

Each GIF runs a scene with the teacher and the trained NCA side by side; the bar underneath is
the nutrient `n`.

In [21]:
def scene_gif(model, cfg, name, n_fn, steps, every=3, seed=0, path=None):
    env = scene_env(cfg, name)
    _, frames = run_world(model, env, n_fn, steps, record_every=every, seed=seed)
    gif = [frame_with_bar(side_by_side(upscale(composite(tr[0], m[0]), 4), upscale(composite(nr[0], m[0]), 4)),
                          float(torch.as_tensor(n_fn(i * every)).reshape(-1)[0]))
           for i, (tr, nr, m) in enumerate(frames)]
    save_gif(gif, path or os.path.join(cfg.out_dir, f"bact_live_{name}.gif"))


for name, n, steps in (("chase", 0.6, 300), ("decoy", 0.6, 200), ("toxin", 0.6, 200),
                       ("trap", 1.0, 250), ("green", 0.6, 90)):
    scene_gif(model, CFG, name, lambda t, n=n: n, steps)
scene_gif(model, CFG, "open", lambda t: 1.0 if t < 150 else 0.2, 260,
          path=os.path.join(CFG.out_dir, "bact_live_retreat.gif"))

## a random crowd, the closest thing to the live demo
crowd_n = lambda t: 0.3 + 0.7 * min(1.0, t / 150)
_env = Env(CFG, World.random(CFG, 1, np.random.default_rng(4242)))
_, _fr = run_world(model, _env, crowd_n, 400, record_every=3, seed=1)
save_gif([frame_with_bar(side_by_side(upscale(composite(tr[0], m[0]), 4), upscale(composite(nr[0], m[0]), 4)),
                         crowd_n(i * 3)) for i, (tr, nr, m) in enumerate(_fr)],
         os.path.join(CFG.out_dir, "bact_live_crowd.gif"))

saved outputs\bact_live_chase.gif (101 frames)
saved outputs\bact_live_decoy.gif (67 frames)
saved outputs\bact_live_toxin.gif (67 frames)
saved outputs\bact_live_trap.gif (84 frames)
saved outputs\bact_live_green.gif (31 frames)
saved outputs\bact_live_retreat.gif (87 frames)
saved outputs\bact_live_crowd.gif (134 frames)


## Live camera -- from an overhead frame to the five masks

**Hardware path:** Sony a6000 -> (HDMI capture card, or Sony *Imaging Edge Webcam*) -> **OBS** ->
*Start Virtual Camera*. Windows then sees "OBS Virtual Camera" as an ordinary webcam, read with
`cv2.VideoCapture(index, cv2.CAP_DSHOW)`. In OBS, crop the scene to the floor area; the notebook's
calibration then fixes the remaining perspective.

**Segmentation, per frame:** warp the floor quad to a square -> blur -> HSV threshold per colour
-> morphological open/close -> drop connected blobs smaller than a torso (`min_area`, in grid
cells) -> area-fraction pool onto the grid -> a cell belongs to a colour if more than `cell_frac`
of it is that colour. Black is only what is dark *and* not another colour.

**Real-world gotchas** (all tunable in `Calib`, saved to `outputs/live_calib.json`):

- *Black hair seen from above* reads as toxin -- the `min_area` filter drops head-sized blobs, so
  toxin people should wear a **large black top**, and everyone else's hair is ignored.
- *Skin, wooden floors, warm light* can read as orange/red -- raise `s_lo` (saturation floor).
- *Shadows* can read as black -- lower black's `v_hi`, keep `min_area` up, light the floor evenly.
- **a6000: set manual white balance and manual exposure.** Auto white balance shifts every colour
  as people move through the frame. A plain, light, matte floor helps more than anything.
- Scale: training people are discs of radius 2-4 cells on a 48-cell grid, i.e. a person is about
  1/10 of the floor width. For ~50 cm shoulders that is a ~4-5 m square floor; for a smaller
  floor, lower `LIVE_GRID` so people stay 4-8 cells wide (the rule is local -- any grid size works).

In [22]:
import cv2


def _default_hsv():
    """Per colour: hue window [h_lo, h_hi] (wraps if h_lo > h_hi), floors on S and V, cap on V.
    OpenCV hue is 0..179."""
    return {
        "red":    dict(h_lo=172, h_hi=8,   s_lo=120, v_lo=70, v_hi=255),
        "orange": dict(h_lo=9,   h_hi=22,  s_lo=140, v_lo=90, v_hi=255),
        "black":  dict(h_lo=0,   h_hi=179, s_lo=0,   v_lo=0,  v_hi=55),
        "blue":   dict(h_lo=95,  h_hi=130, s_lo=100, v_lo=40, v_hi=255),
        "green":  dict(h_lo=40,  h_hi=85,  s_lo=80,  v_lo=50, v_hi=255),
    }


@dataclass
class Calib:
    corners: list = None            ## 4 floor corners [x, y] in camera pixels: TL, TR, BR, BL
    hsv: dict = field(default_factory=_default_hsv)
    floor_lab: list = None
    floor_dist: float = 35.0
    min_chroma: float = 45.0
    black_margin: float = 25.0
    min_area: float = 6.0           ## smallest kept blob, in grid cells (a head is ~2-3)
    cell_frac: float = 0.3          ## a cell is a colour if > this fraction of it is
    warp_px: int = 240              ## side of the warped floor image (5 px per cell at 48)

    def save(self, path):
        with open(path, "w") as f:
            json.dump(asdict(self), f, indent=2)
        print(f"saved {path}")

    @classmethod
    def load(cls, path):
        with open(path) as f:
            d = json.load(f)
        c = cls(**{k: v for k, v in d.items() if k in cls.__dataclass_fields__})
        c.hsv = {**_default_hsv(), **c.hsv}
        return c

def capture_floor_model(cap, calib, n_frames=30, size=60):
    frames = []
    for _ in range(n_frames):
        ok, f = cap.read()
        if ok:
            warped = warp_floor(f, calib)
            lab = cv2.cvtColor(cv2.GaussianBlur(warped, (5, 5), 0), cv2.COLOR_BGR2LAB)
            frames.append(cv2.resize(lab.astype(np.float32), (size, size)))
    calib.floor_lab = np.median(frames, axis=0).tolist()
    return calib

def warp_floor(frame, calib):
    """Camera frame -> square top-down floor image (warp_px x warp_px)."""
    s = calib.warp_px
    if calib.corners is None:
        return cv2.resize(frame, (s, s), interpolation=cv2.INTER_AREA)
    src = np.float32(calib.corners)
    dst = np.float32([[0, 0], [s - 1, 0], [s - 1, s - 1], [0, s - 1]])
    return cv2.warpPerspective(frame, cv2.getPerspectiveTransform(src, dst), (s, s))


def colour_mask(hsv_img, p):
    lo_s, lo_v, hi_v = p["s_lo"], p["v_lo"], p["v_hi"]
    if p["h_lo"] <= p["h_hi"]:
        return cv2.inRange(hsv_img, (p["h_lo"], lo_s, lo_v), (p["h_hi"], 255, hi_v))
    return cv2.inRange(hsv_img, (p["h_lo"], lo_s, lo_v), (179, 255, hi_v)) | \
        cv2.inRange(hsv_img, (0, lo_s, lo_v), (p["h_hi"], 255, hi_v))

def floor_gate(warped_bgr, calib):
    lab = cv2.cvtColor(cv2.GaussianBlur(warped_bgr, (5, 5), 0), cv2.COLOR_BGR2LAB).astype(np.float32)
    floor = cv2.resize(np.array(calib.floor_lab, np.float32), lab.shape[1::-1])
    dist = np.linalg.norm(lab - floor, axis=2)
    chroma = np.linalg.norm(lab[..., 1:] - 128.0, axis=2)
    vivid = (dist > calib.floor_dist) & (chroma > calib.min_chroma)
    dark = floor[..., 0] - lab[..., 0] > calib.black_margin          # pixel darker than floor
    return vivid, dark


_K3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
_K5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))


def segment(frame_bgr, calib, grid, return_warped=False):
    """Camera frame (BGR uint8) -> (5, grid, grid) float32 masks in COLOURS order."""
    warped = warp_floor(frame_bgr, calib)
    hsv_img = cv2.cvtColor(cv2.GaussianBlur(warped, (5, 5), 0), cv2.COLOR_BGR2HSV)
    vivid, dark = floor_gate(warped, calib) if calib.floor_lab is not None else (None, None)
    min_px = calib.min_area * (calib.warp_px / grid) ** 2
    raw = {}
    for name in COLOURS:
        m = colour_mask(hsv_img, calib.hsv[name])
        if name == "black":
            if dark is not None: m &= dark.astype(np.uint8)*255 
        elif vivid is not None: m &= (vivid * 255).astype(np.uint8)
        m = cv2.morphologyEx(cv2.morphologyEx(m, cv2.MORPH_OPEN, _K3), cv2.MORPH_CLOSE, _K5)
        n_lab, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
        keep = np.zeros(n_lab, bool)
        keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= min_px      ## drop heads, specks, shadows
        raw[name] = keep[lab]
    others = raw["red"] | raw["orange"] | raw["blue"] | raw["green"]
    raw["black"] &= ~others                                   ## dark AND not another colour
    out = np.zeros((5, grid, grid), np.float32)
    for i, name in enumerate(COLOURS):
        frac = cv2.resize(raw[name].astype(np.float32), (grid, grid), interpolation=cv2.INTER_AREA)
        out[i] = frac > calib.cell_frac
    return (out, warped) if return_warped else out

In [ ]:
PERSON_BGR = {RED: (40, 40, 210), ORANGE: (30, 140, 245), BLACK: (28, 26, 30),
              BLUE: (200, 90, 40), GREEN: (70, 170, 60)}


class FakeCamera:
    """Renders a (1-slot) World as an overhead camera frame -- floor, clothing colours, and a
    dark *head* on every person (to exercise the head filter). `read()` returns the frame
    and then advances the world, like cv2.VideoCapture; `last_masks` is the ground truth."""

    def __init__(self, world, px=480, heads=True, noise=6, seed=0):
        self.world, self.px, self.heads = world, px, heads
        rng = np.random.default_rng(seed)                   ## a small bank of sensor noise
        self.noise = [rng.normal(0, noise, (px, px, 3)).astype(np.int16) for _ in range(4)] if noise else None
        self.last_masks, self._k = None, 0

    def render(self):
        g, s = self.world.cfg.grid, self.px / self.world.cfg.grid
        img = np.full((self.px, self.px, 3), (196, 202, 204), np.uint8)       ## light floor
        walls = self.world.walls[0].cpu().numpy().astype(np.uint8)
        img[cv2.resize(walls, (self.px, self.px), interpolation=cv2.INTER_NEAREST) > 0] = PERSON_BGR[BLUE]
        w = self.world
        for i in range(w.pos.shape[1]):
            if not w.active[0, i]:
                continue
            y, x = w.pos[0, i].tolist()
            r, c = w.r[0, i].item(), int(w.col[0, i])
            ctr = (int((x + 0.5) * s), int((y + 0.5) * s))
            cv2.circle(img, ctr, int(round(r * s)), PERSON_BGR[c], -1, lineType=cv2.LINE_AA)   ## = the d<=r cell disc
            if self.heads:                                  ## black hair, seen from above
                cv2.circle(img, ctr, int(0.9 * s), (22, 20, 24), -1, lineType=cv2.LINE_AA)
        if self.noise:
            self._k = (self._k + 1) % len(self.noise)
            img = np.clip(img.astype(np.int16) + self.noise[self._k], 0, 255).astype(np.uint8)
        return img

    def read(self):
        self.last_masks = self.world.masks()[0].cpu().numpy()
        frame = self.render()
        self.world.step()
        return True, frame

    def isOpened(self):
        return True

    def release(self):
        pass

In [ ]:
def test_segment_recovers_fake_camera():
    """Every colour, well-separated people (real people don't overlap), walking slowly."""
    rng = np.random.default_rng(21)
    ious = {c: [] for c in range(5)}
    slots = [(10, 10), (10, 24), (10, 38), (32, 14), (32, 34)]
    for trial in range(6):
        cols = rng.permutation(5)
        people = [P(y + rng.uniform(-2, 2), x + rng.uniform(-2, 2), int(rng.integers(2, 5)),
                    COLOURS[c], *rng.uniform(-0.15, 0.15, 2)) for (y, x), c in zip(slots, cols)]
        w = World.scripted(CFG, [people], seeds=[(44, 24)])
        cam = FakeCamera(w, seed=trial)
        for _ in range(3):
            ok, frame = cam.read()
            seg = segment(frame, Calib(), CFG.grid) > 0
            truth = cam.last_masks > 0
            for c in range(5):
                u = (seg[c] | truth[c]).sum()
                if u:
                    ious[c].append((seg[c] & truth[c]).sum() / u)
    worst = {COLOURS[c]: round(float(np.mean(v)), 2) for c, v in ious.items()}
    assert min(worst.values()) >= 0.75, f"mean IoU per colour {worst}"


def test_segment_ignores_heads():
    w = World.scripted(CFG, [[P(24, 24, 3, "orange"), P(10, 10, 3, "green")]], seeds=[(40, 40)])
    ok, frame = FakeCamera(w, heads=True, seed=1).read()
    seg = segment(frame, Calib(), CFG.grid)
    assert seg[BLACK].sum() == 0, f"{seg[BLACK].sum()} black cells from heads"
    assert seg[ORANGE].sum() >= 20 and seg[GREEN].sum() >= 20


def test_warp_corners_round_trip():
    img = np.zeros((300, 400, 3), np.uint8)
    cv2.rectangle(img, (50, 40), (350, 260), (40, 40, 210), -1)     ## the "floor" is red
    cal = Calib(corners=[[50, 40], [350, 40], [350, 260], [50, 260]])
    seg = segment(img, cal, 48)
    assert seg[RED].mean() > 0.95, seg[RED].mean()


def test_calib_json_round_trip():
    import tempfile
    cal = Calib(corners=[[1, 2], [3, 4], [5, 6], [7, 8]], min_area=4.5)
    cal.hsv["red"]["s_lo"] = 99
    path = os.path.join(tempfile.mkdtemp(), "c.json")
    cal.save(path)
    back = Calib.load(path)
    assert back.corners == cal.corners and back.min_area == 4.5 and back.hsv["red"]["s_lo"] == 99


assert not (run_tests("segment") + run_tests("warp") + run_tests("calib_json")), "camera checks failed"

PASS test_segment_recovers_fake_camera (0.9s)
PASS test_segment_ignores_heads (0.1s)
2/2 passed
PASS test_warp_corners_round_trip (0.0s)
1/1 passed
saved C:\Users\krrish\AppData\Local\Temp\tmp1q3t6in9\c.json
PASS test_calib_json_round_trip (0.0s)
1/1 passed


### The live session

`LiveSession.step(frame)` is the whole live loop body: segment -> relax the fields a few
iterations (so the chemical lags people realistically) -> `steps_per_frame` NCA steps -> a
three-panel view (warped camera | what the colony senses | the colony). The colony is reseeded
automatically if it dies out (crushed by a wall, poisoned, or starved at low `n`).
`run_live()` wraps it in an OpenCV window:

| key | action |
|---|---|
| `q` / `Esc` | quit |
| `space` | pause |
| click (right panel) | place the seed there and reseed |
| `r` | reseed at the current seed |
| `+` / `-` | nutrient up / down (biomass) |
| `a` | toggle the auto-ramp (n climbs 0.3 -> 1 at start) |
| `c` | recalibrate (corners + HSV), then continue |
| `f` | capture floor calibration |

OBS can window-capture this view for projection or recording.

In [4]:
class LiveSession:
    def __init__(self, model, cfg, calib, grid=None, steps_per_frame=2, n=0.3, seed_cell=None,
                 panel_px=384):
        self.model, self.cfg, self.calib = model, cfg, calib
        self.g = grid or cfg.grid
        self.dev = next(model.parameters()).device
        self.fields = Fields(cfg, 1, self.g, self.g, self.dev)
        self.masks = torch.zeros(1, 5, self.g, self.g, device=self.dev)
        self.steps_per_frame, self.panel_px = steps_per_frame, panel_px
        self.n, self.auto, self.t = n, True, 0
        self.seed_cell = seed_cell or (self.g // 2, self.g // 2)
        self.state, self.reseeds, self._w, self._wn = None, 0, None, None
        self.warped = None

    def _weights(self):
        if self._wn != self.n:
            self._w, self._wn = self.model.weights_for(torch.tensor([self.n], device=self.dev)), self.n
        return self._w

    def reseed(self, cell=None):
        """New colony at `cell` (or the current seed); moved to a free cell if blocked."""
        if cell is not None:
            self.seed_cell = cell
        blocked = (self.masks[0, BLUE] + self.masks[0, BLACK]).cpu().numpy() > 0
        y, x = self.seed_cell
        if blocked[y, x]:
            free = np.argwhere(~blocked)
            y, x = free[np.argmin(((free - [y, x]) ** 2).sum(1))]
        seed = torch.tensor([[int(y), int(x)]], device=self.dev)
        self.state = seed_state(Teacher(self.cfg, seed, self.masks).rgba(self.masks), self.cfg)
        self.reseeds += 1

    @torch.no_grad()
    def step(self, frame_bgr):
        m, self.warped = segment(frame_bgr, self.calib, self.g, return_warped=True)
        self.masks = torch.from_numpy(m)[None].to(self.dev)
        self.fields.relax(self.masks, self.cfg.init_iters if self.t == 0 else self.cfg.field_iters)
        if self.auto:
            self.n = round(min(1.0, max(self.n, 0.3 + 0.7 * self.t / 300)), 3)
        if self.state is None or self.state[0, 3].sum().item() < 0.5:
            self.reseed()
        kill = torch.maximum(self.masks[:, BLUE:BLUE + 1], self.masks[:, BLACK:BLACK + 1])
        sense, boost = self.fields.sense(self.masks), self.masks[:, GREEN:GREEN + 1]
        for _ in range(self.steps_per_frame):
            self.state = self.model(self.state, self._weights(), sense, kill, boost)
        self.t += 1
        return self.render()

    def render(self):
        P_ = self.panel_px
        cam = cv2.resize(self.warped, (P_, P_), interpolation=cv2.INTER_AREA)
        cam_rgb = cv2.cvtColor(self.warped, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
        la = self.fields.log_attr()[0, 0].cpu().numpy()
        sensed = composite(torch.zeros(4, self.g, self.g), self.masks[0])
        sensed = sensed * (1 - 0.35 * la[..., None]) + 0.35 * la[..., None] * np.array([1.0, 0.95, 0.55])
        colony = composite(self.state[0, :4], self.masks[0], bg=cam_rgb)
        to_bgr = lambda im: cv2.resize(np.ascontiguousarray((np.clip(im, 0, 1)[..., ::-1] * 255).astype(np.uint8)),
                                       (P_, P_), interpolation=cv2.INTER_NEAREST)
        panel = np.concatenate([cam, to_bgr(sensed), to_bgr(colony)], axis=1)
        bar = np.full((28, panel.shape[1], 3), 235, np.uint8)
        cv2.rectangle(bar, (4, 6), (4 + int(self.n * (P_ - 8)), 22), (80, 160, 76), -1)
        cv2.putText(bar, f"n={self.n:.2f}{' auto' if self.auto else ''}  t={self.t}  reseeds={self.reseeds}",
                    (P_ + 8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (40, 40, 40), 1, cv2.LINE_AA)
        return np.concatenate([panel, bar], axis=0)


def open_source(source):
    """int -> a camera index via DirectShow (OBS Virtual Camera); str -> a video file."""
    if isinstance(source, int):
        cap = cv2.VideoCapture(source, cv2.CAP_DSHOW)
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    else:
        cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        raise RuntimeError(f"could not open video source {source!r} -- in OBS, press "
                           f"'Start Virtual Camera', then try indices 0, 1, 2 (list_cameras())")
    return cap


def list_cameras(max_index=6):
    """Probe DirectShow indices; OBS Virtual Camera is usually the last one."""
    found = []
    for i in range(max_index):
        cap = cv2.VideoCapture(i, cv2.CAP_DSHOW)
        ok = cap.isOpened() and cap.read()[0]
        cap.release()
        if ok:
            found.append(i)
    print("working camera indices:", found)
    return found


def calibrate(source, path=None, calib=None, grid=None):
    """Interactive: click the 4 floor corners (TL, TR, BR, BL), then tune HSV per colour.
    Keys: 1-5 pick a colour, 'u' undo a corner, 's' save + exit, 'q' exit without saving."""
    path = path or os.path.join(CFG.out_dir, "live_calib.json")
    grid = grid or CFG.grid
    cal = calib or (Calib.load(path) if os.path.exists(path) else Calib())
    cap = open_source(source) if not hasattr(source, "read") else source
    clicks, win, tb = [], "calibrate: click TL, TR, BR, BL", "hsv"
    cv2.namedWindow(win)
    cv2.setMouseCallback(win, lambda ev, x, y, *_: clicks.append([x, y])
                         if ev == cv2.EVENT_LBUTTONDOWN and len(clicks) < 4 else None)
    cur = ["red"]
    cv2.namedWindow(tb)
    limits = dict(h_lo=179, h_hi=179, s_lo=255, v_lo=255, v_hi=255)

    def load_bars():
        for k, hi in limits.items():
            cv2.setTrackbarPos(k, tb, int(cal.hsv[cur[0]][k]))

    for k, hi in limits.items():
        cv2.createTrackbar(k, tb, int(cal.hsv[cur[0]][k]), hi, lambda v: None)
    saved = False
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        for k in limits:
            cal.hsv[cur[0]][k] = cv2.getTrackbarPos(k, tb)
        view = frame.copy()
        for i, (x, y) in enumerate(clicks):
            cv2.circle(view, (x, y), 6, (0, 255, 255), 2)
            cv2.putText(view, "TL TR BR BL".split()[i], (x + 8, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
        if len(clicks) == 4:
            cal.corners = clicks
        hsv_img = cv2.cvtColor(cv2.GaussianBlur(warp_floor(frame, cal), (5, 5), 0), cv2.COLOR_BGR2HSV)
        raw = colour_mask(hsv_img, cal.hsv[cur[0]])
        seg = segment(frame, cal, grid)[COLOURS.index(cur[0])]
        seg_big = cv2.resize((seg * 255).astype(np.uint8), raw.shape[::-1], interpolation=cv2.INTER_NEAREST)
        cv2.imshow(win, view)
        cv2.imshow(tb, np.concatenate([raw, seg_big], axis=1))
        cv2.setWindowTitle(tb, f"hsv [{cur[0]}]: raw threshold | kept blobs on the grid  (1-5 colour, s save, q quit)")
        key = cv2.waitKey(30) & 0xFF
        if key in (ord("q"), 27):
            break
        if key == ord("u") and clicks:
            clicks.pop()
            cal.corners = None
        if ord("1") <= key <= ord("5"):
            cur[0] = COLOURS[key - ord("1")]
            load_bars()
        if key == ord("f"):
            print("capturing floor model. clear the floor, then hold still...")
            cal = capture_floor_model(cap, cal)
            print("floor model captured")
        if key == ord("s"):
            cal.save(path)
            saved = True
            break
    cv2.destroyWindow(win)
    cv2.destroyWindow(tb)
    if not hasattr(source, "read"):
        cap.release()
    return cal if saved else (Calib.load(path) if os.path.exists(path) else cal)


def run_live(source, model, cfg, calib_path=None, grid=None, window="bacteria live"):
    calib_path = calib_path or os.path.join(cfg.out_dir, "live_calib.json")
    calib = Calib.load(calib_path) if os.path.exists(calib_path) else Calib()
    if calib.corners is None:
        print("no floor corners calibrated -- using the whole frame (press 'c' to calibrate)")
    cap = open_source(source) if not hasattr(source, "read") else source
    sess = LiveSession(model, cfg, calib, grid=grid)
    clicks = []
    cv2.namedWindow(window)
    cv2.setMouseCallback(window, lambda ev, x, y, *_: clicks.append((x, y)) if ev == cv2.EVENT_LBUTTONDOWN else None)
    paused, t_last, fps, img = False, time.time(), 0.0, None
    try:
        while True:
            if not paused or img is None:
                ok, frame = cap.read()
                if not ok:
                    print("video source ended")
                    break
                img = sess.step(frame)
                now = time.time()
                fps = 0.9 * fps + 0.1 / max(now - t_last, 1e-6)
                t_last = now
            show = img.copy()
            cv2.putText(show, f"{fps:4.1f} fps", (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
            cv2.imshow(window, show)
            key = cv2.waitKey(1) & 0xFF
            while clicks:                                  ## click on the colony panel -> seed
                x, y = clicks.pop()
                P_ = sess.panel_px
                if 2 * P_ <= x < 3 * P_ and y < P_:
                    sess.reseed((int(y * sess.g / P_), int((x - 2 * P_) * sess.g / P_)))
            if key in (ord("q"), 27):
                break
            elif key == ord(" "):
                paused = not paused
            elif key == ord("r"):
                sess.reseed()
            elif key in (ord("+"), ord("=")):
                sess.auto, sess.n = False, round(min(1.0, sess.n + 0.05), 2)
            elif key in (ord("-"), ord("_")):
                sess.auto, sess.n = False, round(max(cfg.n_lo, sess.n - 0.05), 2)
            elif key == ord("a"):
                sess.auto = not sess.auto
            elif key == ord("c"):
                sess.calib = calibrate(cap, calib_path, sess.calib, grid=sess.g)
    finally:
        if not hasattr(source, "read"):
            cap.release()
        cv2.destroyWindow(window)
    return sess

In [7]:
## headless smoke test of the WHOLE live path (fake camera -> segment -> fields -> NCA ->
## panel) -- no camera, no window. The fake crowd walks; the colony must live and react.
LIVE_GRID = CFG.grid
_cam = FakeCamera(World.random(CFG, 1, np.random.default_rng(31)), seed=3)
_sess = LiveSession(model, CFG, Calib(), grid=LIVE_GRID)
_t0, _frames = time.time(), []
for _i in range(300):
    _panel = _sess.step(_cam.read()[1])
    if _i % 3 == 0:
        _frames.append(_panel[..., ::-1] / 255.0)
_fps = 300 / (time.time() - _t0)
assert _panel.shape == (_sess.panel_px + 28, 3 * _sess.panel_px, 3), _panel.shape
print(f"live smoke: 300 frames at {_fps:.0f} fps (camera-free), final n={_sess.n}, "
      f"colony mass {_sess.state[0, 3].sum().item():.1f}, reseeds {_sess.reseeds}")
save_gif([cv2.resize(f, (f.shape[1] // 2, f.shape[0] // 2)) for f in _frames],
         os.path.join(CFG.out_dir, "bact_live_smoke.gif"), duration=40)

NameError: name 'CFG' is not defined

### Run it live

1. OBS: add the camera source, crop to the floor, **Start Virtual Camera**.
2. `list_cameras()` -- find OBS's index (usually the highest).
3. `calibrate(CAM_INDEX)` once: click the floor corners TL, TR, BR, BL; press `1`-`5` to pick a
   colour and tune its sliders until the right panel shows exactly the people wearing it; `s` saves.
4. Set `LIVE = True` and run the cell. A video file path works in place of `CAM_INDEX` for rehearsal.

In [8]:
LIVE = True        ## True: open the camera and run the live window
CAM_INDEX = 1       ## OBS Virtual Camera's DirectShow index, or a video file path
LIVE_GRID = 48      ## grid cells across the floor; keep people ~4-8 cells wide

if LIVE:
    list_cameras()                     ## uncomment to find the OBS index
    calibrate(CAM_INDEX)               ## once per setup (or press 'c' in the live window)
    session = run_live(CAM_INDEX, model, CFG, grid=LIVE_GRID)

NameError: name 'cv2' is not defined

## Recipe: adapt this to your own world

- **Different colour meanings** -- the five channels are just masks; change what a channel *does*
  in `Fields._pins` (sources), `Env.inputs` (kill / boost) and `Teacher` (the supervision), then
  retrain. Adding a colour means a sixth mask channel end to end.
- **Stronger / weaker toxins** -- `rep_weight` and `ell_rep`; the teacher and the NCA's sensed
  field change together, then retrain.
- **More or less greedy** -- `wall_perm` controls how leaky walls are to the chemical: at 0 the
  field is a perfect navigation function (never trapped, BFS-like); higher values make traps
  behind walls more common.
- **Bigger colonies** -- `life_max` (biomass at `n = 1`) and `body_r`.
- **A different floor size** -- `LIVE_GRID` only; the rule is local and runs on any grid.

Digital environment, interactive playground-esque presentation | Also began compartmentalizing it into different python files to allow for OOP purposes later.

In [ ]:
TOOLS = [("red", "target"), ("orange", "decoy"), ("black", "toxin"),
         ("blue", "wall"), ("green", "booster"), (None, "erase")]
SB_W = 90


def disc_mask(g, y, x, r):
    yy, xx = np.mgrid[0:g, 0:g]
    return (yy - y) ** 2 + (xx - x) ** 2 <= r * r

class VirtualSession(LiveSession):
    """Same pipeline as LiveSession, masks are from a painted canvas"""


    def __init__(self, model, cfg, grid=None, **kw):
        super().__init__(model, cfg, Calib(), grid=grid, **kw)   ## Calib() is unused, just
        self.canvas = np.zeros((5, self.g, self.g), np.float32)  ## satisfies the parent ctor

    def paint(self, name, gy, gx, r, erase=False):
        gy, gx = int(np.clip(gy, 0, self.g - 1)), int(np.clip(gx, 0, self.g - 1))
        d = disc_mask(self.g, gy, gx, r)
        self.canvas[:, d] = 0.0
        if not erase:
            self.canvas[COLOURS.index(name), d] = 1.0

    def paint_line(self, name, y0, x0, y1, x1, r, erase=False):
        dist = int(round(max(abs(x1 - x0), abs(y1 - y0), 1)))
        for i in range(dist + 1):
            self.paint(name, y0 + i * (y1 - y0) / dist, x0 + i * (x1 - x0) / dist, r, erase)

    def reseed(self, cell=None):
        blocked = (self.canvas[BLUE] + self.canvas[BLACK]) > 0
        if blocked.all():                                         ## guard: board fully walled
            return
        if cell is not None:
            self.seed_cell = cell
        y, x = self.seed_cell
        if blocked[y, x]:
            free = np.argwhere(~blocked)
            y, x = free[np.argmin(((free - [y, x]) ** 2).sum(1))]
        seed = torch.tensor([[int(y), int(x)]], device=self.dev)
        self.state = seed_state(Teacher(self.cfg, seed, self.masks).rgba(self.masks), self.cfg)
        self.reseeds += 1

    @torch.no_grad()
    def step(self):
        self.masks = torch.from_numpy(self.canvas)[None].to(self.dev)
        self.fields.relax(self.masks, self.cfg.init_iters if self.t == 0 else self.cfg.field_iters)
        if self.auto:
            self.n = round(min(1.0, max(self.n, 0.3 + 0.7 * self.t / 300)), 3)
        if self.state is None or self.state[0, 3].sum().item() < 0.5:
            self.reseed()                                          ## always active, auto-respawns
        kill = torch.maximum(self.masks[:, BLUE:BLUE + 1], self.masks[:, BLACK:BLACK + 1])
        sense, boost = self.fields.sense(self.masks), self.masks[:, GREEN:GREEN + 1]
        for _ in range(self.steps_per_frame):
            self.state = self.model(self.state, self._weights(), sense, kill, boost)
        self.t += 1
        return self.render()

    def render(self):
        P_ = self.panel_px
        la = self.fields.log_attr()[0, 0].cpu().numpy()
        world = composite(torch.zeros(4, self.g, self.g), self.masks[0])
        world = world * (1 - 0.35 * la[..., None]) + 0.35 * la[..., None] * np.array([1.0, 0.95, 0.55])
        colony = composite(self.state[0, :4], self.masks[0])      ## checkerboard -- no camera bg
        to_bgr = lambda im: cv2.resize(np.ascontiguousarray((np.clip(im, 0, 1)[..., ::-1] * 255).astype(np.uint8)),
                                       (P_, P_), interpolation=cv2.INTER_NEAREST)
        panel = np.concatenate([to_bgr(world), to_bgr(colony)], axis=1)
        bar = np.full((28, panel.shape[1], 3), 235, np.uint8)
        cv2.rectangle(bar, (4, 6), (4 + int(self.n * (P_ - 8)), 22), (80, 160, 76), -1)
        cv2.putText(bar, f"n={self.n:.2f}{' auto' if self.auto else ''}  t={self.t}  reseeds={self.reseeds}",
                    (P_ + 8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (40, 40, 40), 1, cv2.LINE_AA)
        return np.concatenate([panel, bar], axis=0)


def draw_sidebar(h, selected):
    bar = np.full((h, SB_W, 3), 230, np.uint8)
    row_h = h // len(TOOLS)
    for i, (name, label) in enumerate(TOOLS):
        y0 = i * row_h
        colour = (120, 120, 120) if name is None else tuple(int(c * 255) for c in PERSON_RGB[COLOURS.index(name)][::-1])
        cv2.rectangle(bar, (8, y0 + 8), (SB_W - 8, y0 + row_h - 8), colour, -1)
        if (name or "erase") == selected:
            cv2.rectangle(bar, (3, y0 + 3), (SB_W - 3, y0 + row_h - 3), (0, 0, 0), 2)
        cv2.putText(bar, label, (6, y0 + row_h - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (30, 30, 30), 1, cv2.LINE_AA)
    return bar


def run_virtual(model, cfg, grid=None, window="bacteria virtual"):
    """Keys: 1-6 or sidebar click to pick a tool, drag to paint, space pause, r reseed,
    a toggle auto-n, x clear board, [ / ] brush size, +/- nutrient, q quit."""
    sess = VirtualSession(model, cfg, grid=grid)
    P_, FULL_H = sess.panel_px, sess.panel_px + 28
    tool, brush, drag = ["red"], [2], {"active": False, "prev": None}

    def on_mouse(ev, x, y, flags, _):
        sx = x - SB_W
        if sx < 0:
            if ev == cv2.EVENT_LBUTTONDOWN:
                i = min(y // (FULL_H // len(TOOLS)), len(TOOLS) - 1)
                tool[0] = TOOLS[i][0] or "erase"
            return
        if sx >= P_ or y >= P_:                      ## only the left (world) panel paints
            return
        gy, gx = y * sess.g / P_, sx * sess.g / P_
        if ev == cv2.EVENT_LBUTTONDOWN:
            drag["active"], drag["prev"] = True, (gy, gx)
        elif ev == cv2.EVENT_MOUSEMOVE and drag["active"]:
            py, px = drag["prev"]
            sess.paint_line(tool[0], py, px, gy, gx, brush[0], erase=(tool[0] == "erase"))
            drag["prev"] = (gy, gx)
        elif ev == cv2.EVENT_LBUTTONUP:
            drag["active"] = False

    cv2.namedWindow(window)
    cv2.setMouseCallback(window, on_mouse)
    paused, img = False, None
    try:
        while True:
            if not paused or img is None:
                img = sess.step()
            cv2.imshow(window, np.concatenate([draw_sidebar(FULL_H, tool[0]), img], axis=1))
            key = cv2.waitKey(30) & 0xFF
            if key in (ord("q"), 27):
                break
            elif key == ord(" "):
                paused = not paused
            elif key == ord("r"):
                sess.reseed()
            elif key == ord("a"):
                sess.auto = not sess.auto
            elif key == ord("x"):
                sess.canvas[:] = 0
            elif key == ord("["):
                brush[0] = max(1, brush[0] - 1)
            elif key == ord("]"):
                brush[0] += 1
            elif ord("1") <= key <= ord("6"):
                tool[0] = TOOLS[key - ord("1")][0] or "erase"
            elif key in (ord("+"), ord("=")):
                sess.auto, sess.n = False, round(min(1.0, sess.n + 0.05), 2)
            elif key in (ord("-"), ord("_")):
                sess.auto, sess.n = False, round(max(cfg.n_lo, sess.n - 0.05), 2)
    finally:
        cv2.destroyWindow(window)
    return sess

In [30]:
VIRTUAL = True
if VIRTUAL:
    vsess = run_virtual(model, CFG, grid=LIVE_GRID)

KeyboardInterrupt: 